# Programowanie baz danych — Tutorial
## Optymalizacja zapytań w wielu silnikach: plany wykonania, złączenia, pamięć

**Tryb:** Jupyter + JupySQL — wszystkie zapytania wykonywane na żywo
**SZBD:** PostgreSQL jako silnik podstawowy, uzupełnienia w **Oracle** i **SQL Server**
**Bazy:** Kotki + Kotki rozszerzone + Przykłady dodatkowe (Chinook)

---

## Plan zajęć

- **Plan wykonania:** język, którym posługuje się optymalizator — koszty, statystyki, szacunki kontra rzeczywistość; trzy dialekty odczytu planu (PostgreSQL, Oracle, SQL Server)
- **Złączenia:** trzy algorytmy (Nested Loops, Merge, Hash), kryteria ich wyboru, wskazówki optymalizatora (hinty) w Oracle i SQL Serverze oraz zależność między kosztem a czasem wykonania
- **Pamięć operacyjna:** `work_mem` / PGA / *memory grant* — sortowanie i haszowanie przy niedoborze pamięci RAM (przelewanie na dysk, tzw. *spill*), wpływ kardynalności grup
- **Porady z internetu pod lupą planu:** osiem praktyk z dwóch artykułów (Medium + GeeksforGeeks) zweryfikowanych na podstawie planów wykonania — rozróżnienie faktów, mitów oraz reguł warunkowych
- **Zadania praktyczne**

> **Jak pracować z notatnikiem:**
> 1. Uruchom pięć pierwszych komórek kodu — inicjują one połączenia z bazami danych na potrzeby całych zajęć.
> 2. Przed uruchomieniem każdego polecenia `EXPLAIN`/`EXPLAIN ANALYZE` spróbuj samodzielnie **przewidzieć** kształt planu, a dopiero potem wykonaj komórkę i porównaj wynik.
> 3. W bloku **Porad** najpierw oceń każdą regułę samodzielnie — zapisz swój werdykt (PRAWDA / MIT / NIE WIEM), a następnie uruchom komórkę z planem i zweryfikuj intuicję z dowodem.
> 4. Podane czasy pochodzą z przykładowego uruchomienia i zostały oznaczone tyldą (`~`) w celu wskazania rzędu wielkości; rzeczywiste pomiary mogą różnić się o ok. ±30%.
> 5. **Na obecnych zajęciach świadomie nie tworzymy żadnych indeksów.** Analizujemy wyłącznie zachowanie wynikające z konstrukcji zapytania i mechanizmów silnika. Indeksy są kluczowym elementem optymalizacji, lecz na obecnym etapie utrudniłyby izolację i interpretację wyboru fizycznych operatorów złączeń oraz agregacji — dlatego będą tematem osobnych zajęć.

**Porady** zaczerpnięto z artykułów *„SQL Optimization Techniques — Enhancing Query Performance”* (Medium, DataWithSantosh) oraz *„Best Practices for SQL Query Optimizations”* (GeeksforGeeks).

In [35]:
from dotenv import load_dotenv

# Wczytanie sekretów (PG_CONN, MSSQL_CONN, ORACLE_CONN) z pliku .env
load_dotenv()

True

In [36]:
# JupySQL / SQLAlchemy — połączenia z trzema SZBD
%reload_ext sql

# Wczytanie connection stringów ze zmiennych środowiskowych (z pliku .env)
%sql $PG_CONN --alias pg
%sql $PG_CONN_KOTKI --alias pgkotki
%sql $MSSQL_CONN --alias mssql
%sql $ORACLE_CONN --alias oracle

%config SqlMagic.displaylimit = 100

Connecting to 'pg'

Connecting and switching to connection 'pgkotki'

Connecting and switching to connection 'mssql'

Connecting and switching to connection 'oracle'

## Laboratorium: 1 000 000 incydentów

Podstawowe środowisko testowe stanowi tymczasowa tabela **`incydenty`** — zawierająca milion wierszy w konwencji KOTKI (20 pseudonimów kotów, 12 wrogów, daty z okresu 10 lat) oraz tabela słownikowa **`wrogowie_ref`** (12 wierszy). Baza Chinook zawiera zbyt mało rekordów, aby różnice kosztów były wyraźnie widoczne w pomiarach czasu (znajdują się tam 412 faktury) — na zbiorze milionowym różnice te sięgają setek milisekund.

Dwie celowe cechy generatora danych (istotne w kolejnych blokach dotyczących złączeń oraz pamięci operacyjnej):

- kolumna `pseudo_id` jest **unikalna** (milion różnych wartości) — zapewnia ekstremalną kardynalność grupowania; kolumna `pseudo` przyjmuje 20 wartości, natomiast `szkodliwosc` — 1000;
- kolumna `szkodliwosc` przyjmuje wartość `NULL` w przyblieniu co setny wiersz (`g % 97 = 0`).

Tabele mają charakter tymczasowy (**TEMP**) i istnieją wyłącznie w bieżącej sesji. Bezpośrednio po ich utworzeniu wykonujemy polecenie `ANALYZE` (aktualizacja statystyk dla planera). Tabele celowo **nie posiadają indeksów** — i na obecnych zajęciach żadnych nie zakładamy.

In [37]:
%%sql pgkotki
DROP TABLE IF EXISTS incydenty;
DROP TABLE IF EXISTS wrogowie_ref;

CREATE TEMP TABLE incydenty AS
SELECT 'KOT_' || (g % 20)::text AS pseudo,
       'WROG_' || (g % 12)::text AS imie_wroga,
       DATE '2015-01-01' + (g % 3650) AS data_incydentu,
       'KOC_' || g::text AS pseudo_id,
       g AS id,
       CASE WHEN g % 97 = 0 THEN NULL ELSE (g % 1000) END AS szkodliwosc,
       CASE WHEN g % 7 = 0 THEN 'US'
            ELSE (ARRAY['PL','DE','FR','UK','ES','IT'])[1 + g % 6] END AS kraj
FROM generate_series(1, 1000000) g;

CREATE TEMP TABLE wrogowie_ref AS
SELECT 'WROG_' || n::text AS imie_wroga, (n % 10) + 1 AS stopien
FROM generate_series(0, 11) n;

ANALYZE incydenty;
ANALYZE wrogowie_ref;

SELECT COUNT(*) AS incydentow,
       COUNT(szkodliwosc) AS z_szkodliwoscia,
       COUNT(DISTINCT pseudo_id) AS unikalnych_pseudo_id,
       COUNT(DISTINCT szkodliwosc) AS wartosci_szkodliwosci
FROM incydenty


Switching to connection 'pgkotki'

1000000 rows affected.

12 rows affected.

1 rows affected.

incydentow,z_szkodliwoscia,unikalnych_pseudo_id,wartosci_szkodliwosci
1000000,989691,1000000,1000


In [9]:
%%sql oracle
-- Schemat = użytkownik w Oracle; nasz Chinook należy do C##CHINOOK.
-- Ustawiamy go raz dla całej sesji — dalsze zapytania są krótsze.
ALTER SESSION SET CURRENT_SCHEMA = C##CHINOOK
;


Switching to connection 'oracle'

++
||
++
++

# Plan wykonania: język optymalizatora

**Pytanie bloku:** gdy napiszesz zapytanie, kto dokładnie decyduje o sposobie jego realizacji — i czy można przeanalizować tę decyzję, zanim zapytanie rozpocznie przetwarzanie danych?

## SQL jest językiem deklaratywnym

Zapytanie SQL określa, **co** chcemy uzyskać, a nie **jak** należy to obliczyć. Silnik bazy danych nie wykonuje instrukcji SQL linijka po linijce — jest to język **deklaratywny**. Za wybór konkretnego sposobu obliczenia wyniku odpowiada **optymalizator zapytań** (ang. *query optimizer*). Ten sam wynik logiczny można uzyskać na wiele sposobów, a różnica kosztu między planem optymalnym a nieoptymalnym potrafi wynosić kilka rzędów wielkości.

```
Zapytanie SQL
   │
   ▼
[1] Analiza leksykalna i składniowa (parser)      → drzewo składni
   │
   ▼
[2] Analiza semantyczna (walidacja)               → sprawdzenie tabel, kolumn, typów, uprawnień
   │
   ▼
[3] Przekształcenie do algebry relacji            → logiczne drzewo zapytania (kanoniczne)
   │
   ▼
[4] Optymalizacja logiczna (reguły równoważności) → zoptymalizowane drzewo logiczne
   │
   ▼
[5] Optymalizacja fizyczna (algorytmy, indeksy, koszty, statystyki) → plan fizyczny
   │
   ▼
[6] Wykonanie planu (silnik wykonawczy)           → wynik
```

## Algebra relacji: operatory

Algebra relacji dostarcza zbioru operatorów, które przyjmują jedną lub dwie relacje i zwracają nową relację. Na ich podstawie budowane są wyrażenia algebraiczne, które można przedstawić w postaci drzew.

| Operator | Symbol | Znaczenie | Odpowiednik w SQL |
| --- | --- | --- | --- |
| Selekcja | `σ_warunek(R)` | wybiera krotki spełniające warunek | `WHERE` (oraz `HAVING` dla grup po agregacji) |
| Projekcja | `π_atrybuty(R)` | wybiera kolumny | lista `SELECT` |
| Eliminacja duplikatów | `δ(R)` | usuwa powtarzające się krotki | `DISTINCT` |
| Iloczyn kartezjański | `R × S` | wszystkie pary krotek | `CROSS JOIN`, przecinek w `FROM` |
| Złączenie θ | `R ⋈_θ S` | iloczyn z selekcją θ | `JOIN ... ON` |
| Złączenie naturalne | `R ⋈ S` | równość wspólnych atrybutów | `NATURAL JOIN` |
| Złączenie zewnętrzne | `R ⟕ S, R ⟖ S, R ⟗ S` | zachowuje krotki bez dopasowania | `LEFT/RIGHT/FULL JOIN` |
| Suma | `R ∪ S` | krotki z obu relacji | `UNION` |
| Przecięcie | `R ∩ S` | krotki wspólne | `INTERSECT` |
| Różnica | `R − S` | krotki z R nieobecne w S | `EXCEPT / MINUS` |
| Przemianowanie | `ρ_x(R)` | zmiana nazwy relacji lub atrybutu | alias (`AS`) |
| Grupowanie i agregacja | `γ_{grupy; agregaty}(R)` | grupuje krotki i oblicza agregaty | `GROUP BY` + `SUM`, `AVG`, `COUNT` |
| Sortowanie | `τ_atrybuty(R)` | porządkuje krotki | `ORDER BY` |

Uwaga: klasyczna algebra relacji operuje na **zbiorach**, natomiast SQL na **multizbiorach** (duplikaty są dozwolone). Dlatego w praktyce optymalizatory stosują algebrę multizbiorów z jawnym operatorem `δ` dla `DISTINCT`.

Rozważmy prosty schemat bazy danych (uproszczony system dziekanatu):

```
Student(id, imie, nazwisko, kierunek_id)
Kurs(id, nazwa, ects)
Zapis(student_id, kurs_id, ocena)
```

Zapytanie A: nazwiska studentów oraz nazwy kursów, z których uzyskali oni ocenę powyżej 3,0, dla przedmiotów o wartości co najmniej 5 punktów ECTS.

```sql
SELECT s.nazwisko, k.nazwa
FROM Student s, Zapis z, Kurs k
WHERE s.id = z.student_id
  AND z.kurs_id = k.id
  AND z.ocena > 3.0
  AND k.ects >= 5;
```

Bezpośrednie tłumaczenie zgodne z kolejnością klauzul SQL obejmuje najpierw `FROM` (iloczyn kartezjański), następnie `WHERE` (selekcja), a na końcu `SELECT` (projekcja).

Wyrażenie algebraiczne:

```
π_{s.nazwisko, k.nazwa} (
  σ_{s.id = z.student_id ∧ z.kurs_id = k.id ∧ z.ocena > 3.0 ∧ k.ects ≥ 5} (
    (Student s × Zapis z) × Kurs k
  )
)
```

Warto zaznaczyć, że w algebrze relacji ten sam cel można zrealizować za pomocą wielu równoważnych wyrażeń. Wybór odpowiedniego wyrażenia ma kluczowe znaczenie dla wydajności ze względu na jego złożoność obliczeniową oraz zapotrzebowanie na pamięć.

Zapytanie to można również przedstawić jako **drzewo zapytania**:
- **Liście** to relacje bazowe (tabele), ewentualnie z przemianowaniem `ρ`.
- **Węzły wewnętrzne** to operatory algebry relacji, w tym operatory jednoargumentowe (`σ`, `π`, `γ`, `τ`, `δ`) oraz binarne (`×`, `⋈`, `∪`, `∩`, `−`).
- **Krawędzie** reprezentują przepływ krotek — dane płyną od liści ku korzeniowi.
- **Korzeń** zwraca końcowy wynik zapytania.

Poniżej przedstawiono kanoniczne drzewo zapytania dla omawianego przykładu:
```mermaid
graph TD
    N1["π s.nazwisko, k.nazwa"] --> N2["σ s.id = z.student_id ∧ z.kurs_id = k.id ∧ z.ocena > 3.0 ∧ k.ects ≥ 5"]
    N2 --> N3["×"]
    N3 --> N4["×"]
    N3 --> N5["Kurs k"]
    N4 --> N6["Student s"]
    N4 --> N7["Zapis z"]
```

Zapis liniowy (wyrażenie algebraiczne) oraz drzewo stanowią dwie równoważne reprezentacje tego samego obiektu.

> **Reguły równoważności algebraicznej:** Optymalizator stosuje reguły przekształcające drzewo na postać równoważną (zwracającą identyczny wynik). Najważniejsze z nich to:
> 1. **Rozbicie selekcji koniunkcyjnej (kaskada selekcji):** $\sigma_{c_1 \land c_2}(R) \equiv \sigma_{c_1}(\sigma_{c_2}(R))$.
> 2. **Przemienność selekcji:** $\sigma_{c_1}(\sigma_{c_2}(R)) \equiv \sigma_{c_2}(\sigma_{c_1}(R))$.
> 3. **Przesunięcie selekcji w dół drzewa (ang. *selection pushdown*):** jeśli warunek $c$ dotyczy wyłącznie atrybutów relacji $R$, to $\sigma_c(R owtie S) \equiv \sigma_c(R) owtie S$. Redukuje to liczbę przetwarzanych krotek na najwcześniejszym możliwym etapie i jest zazwyczaj najbardziej opłacalną regułą optymalizacji.
> 4. **Połączenie selekcji z iloczynem kartezjańskim w złączenie:** $\sigma_	heta(R 	imes S) \equiv R owtie_	heta S$, o ile predykat $	heta$ łączy atrybuty obu relacji.
> 5. **Przemienność złączeń i iloczynów:** $R owtie S \equiv S owtie R$.
> 6. **Łączność złączeń:** $(R owtie S) owtie T \equiv R owtie (S owtie T)$. W połączeniu z regułą 5 pozwala to wybrać dowolną kolejność wykonywania złączeń.
> 7. **Kaskada projekcji i przesunięcie projekcji w dół (ang. *projection pushdown*):** zachowujemy wyłącznie atrybuty niezbędne na wyższych poziomach drzewa (w wyniku końcowym oraz w warunkach złączeń), co zmniejsza szerokość krotek.
> 8. **Przemienność selekcji z projekcją**, pod warunkiem że projekcja nie usuwa atrybutów wykorzystywanych w predykacie selekcji.
>
> **Uwaga dotycząca złączeń zewnętrznych:** reguły 5 i 6 oraz swobodne przesuwanie selekcji **nie zawsze** obowiązują dla `LEFT/RIGHT/FULL JOIN`. Optymalizator musi uwzględnić, po której stronie zachowywane są krotki niedopasowane.

Po zastosowaniu reguł 1, 3, 4 i 7 względem drzewa kanonicznego eliminujemy iloczyny kartezjańskie, selekcje wykonywane są bezpośrednio przy odczycie danych z tabel, a w górę drzewa przekazywane są wyłącznie wymagane kolumny:
```mermaid
graph TD
    A["π s.nazwisko, k.nazwa"] --> B["⋈ z.kurs_id = k.id"]
    
    B --> C["⋈ s.id = z.student_id"]
    B --> H["π id, nazwa"]
    
    C --> D["π id, nazwisko"]
    C --> E["π student_id, kurs_id"]
    
    D --> F["Student s"]
    
    E --> G["σ ocena > 3.0"]
    G --> I["Zapis z"]
    
    H --> J["σ ects ≥ 5"]
    J --> K["Kurs k"]
```

Alternatywnie optymalizator mógłby opóźnić złączenie z tabelą `Student` i w pierwszej kolejności połączyć relacje `Zapis` oraz `Kurs`:
```mermaid
graph TD
    N1["π s.nazwisko, k.nazwa"] --> N2["⋈ s.id = z.student_id"]
    N2 --> N3["Student s"]
    N2 --> N4["⋈ z.kurs_id = k.id"]
    N4 --> N5["σ ocena > 3.0"]
    N5 --> N6["Zapis z"]
    N4 --> N7["σ ects ≥ 5"]
    N7 --> N8["Kurs k"]
```

Przedstawione drzewo opisuje logiczną strukturę zapytania. Aby je wykonać, niezbędne jest przejście na warstwę fizyczną, w której poszczególnym operatorom algebry relacji przypisywane są konkretne algorytmy. Bezpośrednia, naiwna realizacja drzewa kanonicznego byłaby **poprawna semantycznie, lecz skrajnie kosztowna**: przykładowo iloczyn kartezjański generuje $|Student| 	imes |Zapis| 	imes |Kurs|$ krotek pośrednich. Zadaniem optymalizatora jest zatem przekształcenie drzewa logicznego w równoważny, najtańszy fizyczny plan wykonania.

## Co mogłoby decydować o preferencjach w kolejności złączeń?

```{toggle} P
Wybór między takimi wariantami zależy od **statystyk** oraz **modelu kosztów**.
```

## Co to jest plan wykonania i skąd się bierze

**Optymalizator zapytań** (ang. *query optimizer*) decyduje, *jak* obliczyć wynik zapytania. Rezultatem jego pracy jest **plan wykonania** (ang. *execution plan*) — drzewo operatorów fizycznych, takich jak `Seq Scan` (pełny skan tabeli), `Index Scan` (odczyt z wykorzystaniem indeksu), `Sort`, `Hash Join` czy `Aggregate`. Plan analizuje się **od dołu do góry**: liście odpowiadają za odczyt danych ze źródeł, każdy wyższy węzeł przetwarza strumień danych z węzła podrzędnego, a korzeń zwraca wynik końcowy.

Większość współczesnych optymalizatorów działa w oparciu o model kosztowy (ang. *Cost-Based Optimizer*, CBO) i wyznacza **koszt** każdego kandydata na podstawie dwóch filarów:

1. **Statystyk** — zapisanego w katalogu systemowym opisu danych: liczby wierszy w tabeli, liczby unikalnych wartości w kolumnie, odsetka wartości `NULL` (w PostgreSQL: `pg_class.reltuples`, `pg_stats`).
2. **Modelu kosztu** — zestawu wzorów przypisujących każdemu operatorowi koszt wyrażony w umownych **jednostkach**; optymalizator sumuje te jednostki i wybiera plan o najniższym koszcie całkowitym. W przypadku rozbudowanych zapytań (gdzie liczba możliwych planów rośnie wykładniczo) przestrzeń poszukiwań ograniczana jest za pomocą programowania dynamicznego lub heurystyk.

**Jednostka kosztu nie odpowiada milisekundom.** W PostgreSQL bazową jednostką jest koszt odczytu pojedynczej strony w trybie sekwencyjnym (`seq_page_cost = 1.0`); w Oracle jest to jednostka pojedynczej operacji wejścia/wyjścia (*single-block I/O*); w SQL Serverze — wewnętrzna skala estymacji. Wartości kosztu pozwalają **porównywać alternatywne plany w obrębie jednego silnika bazodanowego**, lecz nie można ich bezpośrednio przeliczać między różnymi SZBD ani na czas rzeczywisty.

### Jak uzyskać plan wykonania w PostgreSQL

Podstawowym narzędziem diagnostycznym jest **plan wykonania**, w którym optymalizator krok po kroku opisuje sposób realizacji zapytania.

Plan ten uzyskuje się za pomocą polecenia `EXPLAIN`.

Polecenie `EXPLAIN` przeprowadza **symulację**: optymalizator wyznacza plan i szacuje koszty, ale samo zapytanie nie jest uruchamiane. Natomiast `EXPLAIN ANALYZE` wykonuje **rzeczywisty pomiar**: zapytanie jest fizycznie uruchamiane (dotyczy to również instrukcji `UPDATE` i `DELETE` — należy zachować szczególną ostrożność w środowiskach produkcyjnych, stosując np. blok transakcji z `ROLLBACK`), a do planu dołączane są rzeczywiste czasy wykonania oraz liczby wierszy przetworzonych przez każdy węzeł.

In [38]:
%%sql pgkotki
-- EXPLAIN: przedstawia samo oszacowanie (plan, którego optymalizator oczekuje), nic nie jest wykonywane
EXPLAIN
SELECT
  pseudo,
  COUNT(*)
FROM
  incydenty
GROUP BY
  pseudo

3 rows affected.

QUERY PLAN
HashAggregate (cost=24408.00..24408.20 rows=20 width=14)
Group Key: pseudo
-> Seq Scan on incydenty (cost=0.00..19408.00 rows=1000000 width=6)


In [11]:
%%sql pgkotki
-- 2) EXPLAIN ANALYZE: plan + RZECZYWISTOŚĆ (zapytanie naprawdę się wykonuje!)
--    actual time/rows/loops dostępne są jako dodatkowe informacje
EXPLAIN ANALYZE
SELECT
  pseudo,
  COUNT(*)
FROM
  incydenty
GROUP BY
  pseudo

8 rows affected.

QUERY PLAN
HashAggregate (cost=24408.00..24408.20 rows=20 width=14) (actual time=252.714..252.717 rows=20.00 loops=1)
Group Key: pseudo
Batches: 1 Memory Usage: 32kB
Buffers: local read=9408
-> Seq Scan on incydenty (cost=0.00..19408.00 rows=1000000 width=6) (actual time=0.045..92.874 rows=1000000.00 loops=1)
Buffers: local read=9408
Planning Time: 0.051 ms
Execution Time: 252.767 ms


In [6]:
%%sql pgkotki
-- 3) EXPLAIN (ANALYZE, BUFFERS): + fizyczne strony (hit = cache, read = dysk)
--    sort/hash raportują tu swój tryb pracy (Sort Method / Batches).
EXPLAIN (ANALYZE, BUFFERS)
SELECT
  pseudo,
  COUNT(*)
FROM
  incydenty
GROUP BY
  pseudo
ORDER BY
  pseudo

12 rows affected.

QUERY PLAN
Sort (cost=24408.63..24408.68 rows=20 width=14) (actual time=248.759..248.760 rows=20.00 loops=1)
Sort Key: pseudo
Sort Method: quicksort Memory: 25kB
Buffers: local read=9408
-> HashAggregate (cost=24408.00..24408.20 rows=20 width=14) (actual time=248.724..248.726 rows=20.00 loops=1)
Group Key: pseudo
Batches: 1 Memory Usage: 32kB
Buffers: local read=9408
-> Seq Scan on incydenty (cost=0.00..19408.00 rows=1000000 width=6) (actual time=0.040..94.249 rows=1000000.00 loops=1)
Buffers: local read=9408


In [7]:
%%sql pgkotki
-- 4) SETTINGS (aktywne parametry planera) i WAL USAGE.
--    Uwaga: dla zwykłego SELECT-a na tabeli TEMP sekcja SETTINGS może być
--    pusta — parametry wpływające pojawiają się tylko wtedy, gdy odbiegają
--    od domyślnych W TEJ sesji.
EXPLAIN (
  ANALYZE,
  BUFFERS,
  SETTINGS,
  WAL,
  TIMING ON,
  SUMMARY ON,
  FORMAT TEXT
)
SELECT
  COUNT(*)
FROM
  incydenty

6 rows affected.

QUERY PLAN
Aggregate (cost=21908.00..21908.01 rows=1 width=8) (actual time=130.384..130.384 rows=1.00 loops=1)
Buffers: local read=9408
-> Seq Scan on incydenty (cost=0.00..19408.00 rows=1000000 width=0) (actual time=0.042..91.652 rows=1000000.00 loops=1)
Buffers: local read=9408
Planning Time: 0.034 ms
Execution Time: 130.404 ms


Wszystkie cztery warianty prezentują to samo drzewo operatorów, różniąc się zakresem **informacji diagnostycznych o rzeczywistym wykonaniu**:

| Wariant | Zakres informacji | Zastosowanie |
|---|---|---|
| `EXPLAIN` | szacunki planera: `cost=..rows=..width=..` | szybka weryfikacja strategii optymalizatora |
| `EXPLAIN ANALYZE` | + `actual time`, `actual rows`, `loops`, `Execution Time` | weryfikacja trafności estymacji, identyfikacja wąskich gardeł |
| `EXPLAIN (ANALYZE, BUFFERS)` | + statystyki operacji na stronach pamięci (`shared` / `local` / `temp`) | analiza operacji dyskowych względem pamięci podręcznej (*cache*) |
| `+ SETTINGS, WAL, TIMING` | + niestandardowe parametry sesji, zapisy do dziennika WAL, czasy per węzeł | zaawansowana diagnostyka produkcyjna |

**Interpretacja linii węzła** na przykładzie operatora skanowania:

```
Seq Scan on incydenty  (cost=0.00..19408.00 rows=1000000 width=17)
                        (actual time=0.015..120.324 rows=1000000 loops=1)
```

- `cost=A..B` — **dwie wartości**: koszt uruchomienia do zwrócenia pierwszego wiersza (*startup cost*) oraz koszt całkowity do zwrócenia ostatniego wiersza (*total cost*). Przy porównywaniu planów analizuje się przede wszystkim koszt całkowity;
- `rows` (szacunek) vs `actual rows` (rzeczywistość) — **kluczowy wskaźnik błędów estymacji**: jeśli planer myli się o rząd wielkości, błąd ten będzie propagowany i multiplikowany w wyższych węzłach planu (zwłaszcza w złączeniach);
- `width` — szacowana średnia szerokość wiersza opuszczającego dany węzeł (w bajtach); zmniejsza się po zastosowaniu wąskiej projekcji (ograniczeniu liczby kolumn);
- `loops` — liczba wywołań danego węzła; wartość `actual time` wyraża średni czas **pojedynczego** wykonania (łączny czas pracy węzła to iloczyn `actual time × loops`).

Zwróć uwagę na różnicę między operatorem `Seq Scan` na tabeli `incydenty` (pełny skan tabeli) a węzłem `Sort` w trzecim wariancie. Sortowanie dotyczy tu niewielkiego zbioru, jednak faktyczne zużycie pamięci jest raportowane dopiero przy użyciu opcji `(ANALYZE, BUFFERS)` w linii `Sort Method`.

Kolejne wcięcia wyznaczają strukturę **drzewa operatorów**, które analizuje się od liści (najbardziej zagnieżdżonych węzłów) ku korzeniowi.

Trzy podstawowe pytania diagnostyczne przy analizie **każdego** planu wykonania:

1. **Kształt drzewa: jaki węzeł stanowi liść?** Operator `Seq Scan` oznacza odczyt całej tabeli od początku do końca — na tabeli nieposiadającej indeksów jest to standardowa metoda dostępu.
2. **Liczebność: czy wartość `rows` jest zgodna z `actual rows`?** Znaczna rozbieżność świadczy o nieaktualnych statystykach lub trudnych do oszacowania predykatach.
3. **Skala obciążenia: gdzie koncentruje się koszt?** Wąskie gardło uwidacznia się jako węzeł o największym przyroście kosztu całkowitego, czasu wykonania lub liczby odczytanych buforów.

## Operatory

### Operatory dostępu do danych (liście planu)

| Operator | PostgreSQL | SQL Server | Oracle | Kiedy się opłaca | Uwagi |
|---|---|---|---|---|---|
| Pełny skan tabeli | `Seq Scan` | `Table Scan` / `Clustered Index Scan` | `TABLE ACCESS FULL` | duża część tabeli (niska selektywność), mała tabela, brak indeksu | czyta wszystkie strony sekwencyjnie, najtańszy wzorzec I/O |
| Wyszukiwanie po indeksie | `Index Scan` | `Index Seek` + `Key/RID Lookup` | `INDEX RANGE SCAN` / `UNIQUE SCAN` + `TABLE ACCESS BY ROWID` | wysoka selektywność (warunek `=` lub wąski zakres) | każdy trafiony wiersz może wymagać dodatkowego odczytu z tabeli |
| Skan samego indeksu | `Index Only Scan` | `Index Seek/Scan` (indeks pokrywający) | `INDEX FAST FULL SCAN` / `RANGE SCAN` bez dostępu do tabeli | indeks zawiera wszystkie potrzebne kolumny | unika odczytu stron tabeli |
| Skan bitmapowy | `Bitmap Index Scan` + `Bitmap Heap Scan` | łączenie indeksów (*Index Intersection*) | `BITMAP INDEX` / `BITMAP CONVERSION` | średnia selektywność, łączenie wielu indeksów (`AND`/`OR`) | zbiera adresy stron, a następnie odczytuje je w kolejności fizycznej |
| Dostęp po adresie wiersza | `TID Scan` | `RID Lookup` | `TABLE ACCESS BY ROWID` | znany fizyczny adres wiersza | zwykle element większego planu |
| Skan stałych wartości | `Values Scan` / `Result` | `Constant Scan` | `FAST DUAL` / `COUNT` | `VALUES (...)`, `SELECT 1` | brak tabeli źródłowej |

### Operatory złączenia

| Algorytm | Nazwa w PostgreSQL / SQL Server / Oracle | Wymagania | Typowy koszt | Kiedy najlepszy | Wada |
|---|---|---|---|---|---|
| Pętle zagnieżdżone | `Nested Loop` / `Nested Loops` / `NESTED LOOPS` | dowolny warunek (także nierówności) | $O(n \cdot m)$ bez indeksu, $O(n \cdot \log m)$ z indeksem po stronie wewnętrznej | małe wejście zewnętrzne, indeks na wewnętrznym, zapytania OLTP | bardzo kosztowny dla dużych wejść bez indeksu |
| Złączenie haszowe | `Hash Join` / `Hash Match` / `HASH JOIN` | wyłącznie warunek równościowy | $O(n + m)$ | duże, nieposortowane wejścia, zapytania analityczne | tablica haszująca zajmuje pamięć (przy jej braku dane trafiają na dysk); operator blokujący po stronie budowania |
| Złączenie scalające | `Merge Join` / `Merge Join` / `MERGE JOIN (SORT JOIN)` | wejścia posortowane po kluczu, warunek równościowy lub zakresowy | $O(n + m)$ plus ewentualne sortowanie | wejścia już posortowane (indeksy), wynik ma być posortowany | narzut sortowania przy braku wcześniejszego uporządkowania |
| Iloczyn kartezjański | `Nested Loop` bez warunku / `Nested Loops` / `MERGE JOIN CARTESIAN` | brak | $O(n \cdot m)$ | małe relacje, celowy `CROSS JOIN` | nieoczekiwane pojawienie się w planie zwykle oznacza brak warunku złączenia |

### Operatory agregacji, sortowania i eliminacji duplikatów

| Operator logiczny | Operator fizyczny | PostgreSQL / SQL Server / Oracle | Wymaga sortowania | Blokujący | Kiedy |
|---|---|---|---|---|---|
| $\gamma$ (grupowanie) | agregacja haszowa | `HashAggregate` / `Hash Match (Aggregate)` / `HASH GROUP BY` | nie | tak | dużo grup, dane nieposortowane |
| $\gamma$ (grupowanie) | agregacja strumieniowa | `GroupAggregate` / `Stream Aggregate` / `SORT GROUP BY` | tak (lub dane z indeksu) | częściowo (grupa po grupie) | dane już uporządkowane po kluczu grupowania |
| $\gamma$ (agregat bez `GROUP BY`) | agregat skalarny | `Aggregate` / `Stream Aggregate` / `SORT AGGREGATE` | nie | tak | `COUNT(*)`, `SUM()` dla całej relacji |
| $	au$ (sortowanie) | sortowanie w pamięci | `Sort (quicksort)` / `Sort` / `SORT ORDER BY` | n/d | tak | zbiór mieści się w pamięci roboczej |
| $	au$ (sortowanie) | sortowanie zewnętrzne | `Sort (external merge)` / `Sort` (spill do `tempdb`) / `SORT ORDER BY` (na dysku) | n/d | tak | duże zbiory danych, niewystarczająca pamięć robocza |
| $	au + 	ext{LIMIT}$ | sortowanie typu top-N | `Sort (top-N heapsort)` / `Top N Sort` / `SORT ORDER BY STOPKEY` | n/d | tak | `ORDER BY ... LIMIT n` |
| $\delta$ (`DISTINCT`) | eliminacja haszowa | `HashAggregate` / `Hash Match (Flow Distinct)` / `HASH UNIQUE` | nie | tak | duże dane, nieposortowane |
| $\delta$ (`DISTINCT`) | eliminacja po sortowaniu | `Unique` (po `Sort`) / `Sort (Distinct Sort)` / `SORT UNIQUE` | tak | tak | dane już posortowane lub wymagany wynik uporządkowany |

### Operatory zbiorowe i pomocnicze

| Operator | Znaczenie | PostgreSQL / SQL Server / Oracle | Blokujący | Uwagi |
|---|---|---|---|---|
| Suma bez eliminacji duplikatów | `UNION ALL` | `Append` / `Concatenation` / `UNION-ALL` | nie | najtańsza operacja zbiorowa |
| Suma z eliminacją | `UNION` | `Append` + `HashAggregate` lub `Unique` / `Concatenation` + `Distinct` / `SORT UNIQUE` | tak | dodatkowy koszt eliminacji duplikatów |
| Przecięcie, różnica | `INTERSECT`, `EXCEPT` | `HashSetOp` / `SetOp` / `Hash Match (Left Semi/Anti)` / `INTERSECTION`, `MINUS` | tak | często realizowane jako półzłączenie lub antyzłączenie |
| Filtr | selekcja $\sigma$ nad wynikiem operatora | `Filter` / `Filter` / `FILTER` | nie | zwykle wciśnięty do skanu, osobny węzeł dla warunków nieprzesuniętych |
| Obliczenie wyrażeń | projekcja $\pi$ z wyrażeniami | `Result` / `Compute Scalar` / kolumny wyliczane | nie | brak kosztu I/O |
| Ograniczenie wyniku | `LIMIT` / `TOP` / `FETCH FIRST` | `Limit` / `Top` / `COUNT STOPKEY` | nie | pozwala zatrzymać potok po $n$ krotkach |
| Materializacja | zapamiętanie wyniku pośredniego do wielokrotnego użycia | `Materialize` / `Table Spool`, `Eager Spool` / `TEMP TABLE TRANSFORMATION` | tak | używana np. po stronie wewnętrznej `Nested Loop` |
| Podplan, CTE | wykonanie podzapytania | `SubPlan`, `CTE Scan` / `Apply`, `Spool` / `VIEW`, `FILTER` | zależnie od przypadku | podzapytania skorelowane wykonywane wielokrotnie są kosztowne |
| Funkcje okienkowe | `OVER (PARTITION BY ... ORDER BY ...)` | `WindowAgg` / `Window Spool`, `Segment`, `Sequence Project` / `WINDOW SORT` | zwykle tak | wymaga uporządkowania po kluczach partycji i sortowania |
| Wykonanie równoległe | zbieranie wyników od wielu procesów | `Gather`, `Gather Merge` / `Parallelism (Gather Streams)` / `PX COORDINATOR` | n/d | koszt koordynacji, opłaca się dla dużych zbiorów danych |

### Preferowany wybór operatora

| Sytuacja | Preferowany operator |
|---|---|
| Mała tabela lub odczyt większości wierszy z tabeli | pełny skan (`Seq Scan` / `Table Scan` / `TABLE ACCESS FULL`) |
| Warunek `=` na kolumnie z indeksem, potrzebne dodatkowe kolumny i mało wyników | skan indeksu (`Index Seek` / `Index Scan`) |
| Indeks zawiera wszystkie potrzebne kolumny, brak odwołań do kolumn nieposiadających indeksu | skan samego indeksu (`Index Only Scan`) |
| Małe wejście zewnętrzne i indeks na wejściu wewnętrznym | `Nested Loop` |
| Dwa duże wejścia, złączenie równościowe, brak uporządkowania | `Hash Join` |
| Wejścia już posortowane lub wymagany wynik uporządkowany | `Merge Join` |
| Złączenie z warunkiem nierównościowym | `Nested Loop` (lub `Merge Join` dla zakresów) |
| Dużo grup, dane nieposortowane | agregacja haszowa (`HashAggregate`) |
| Dane już uporządkowane po kluczu grupowania | agregacja strumieniowa (`GroupAggregate`) |
| `ORDER BY ... LIMIT n` | sortowanie top-N lub odczyt z indeksu w żądanej kolejności |

Nazwy operatorów różnią się między systemami i ich kolejnymi wersjami, dlatego w razie wątpliwości należy odwołać się do dokumentacji konkretnego SZBD.

In [24]:
%%sql pgkotki

EXPLAIN ANALYZE
SELECT
  i.pseudo,
  wg.imie_wroga,
  COUNT(*)
FROM
  incydenty i LEFT JOIN wrogowie_ref wg ON i.imie_wroga = wg.imie_wroga
WHERE stopien = 7
GROUP BY
  i.pseudo, wg.imie_wroga

18 rows affected.

QUERY PLAN
HashAggregate (cost=24617.49..24617.69 rows=20 width=21) (actual time=186.367..186.370 rows=5.00 loops=1)
"Group Key: i.pseudo, wg.imie_wroga"
Batches: 1 Memory Usage: 32kB
Buffers: local read=9409
-> Hash Join (cost=1.16..23992.49 rows=83333 width=13) (actual time=0.060..169.837 rows=83333.00 loops=1)
Hash Cond: (i.imie_wroga = wg.imie_wroga)
Buffers: local read=9409
-> Seq Scan on incydenty i (cost=0.00..19408.00 rows=1000000 width=13) (actual time=0.040..89.751 rows=1000000.00 loops=1)
Buffers: local read=9408
-> Hash (cost=1.15..1.15 rows=1 width=7) (actual time=0.015..0.016 rows=1.00 loops=1)


In [25]:
%%sql pgkotki

EXPLAIN ANALYZE
SELECT
  i.pseudo,
  wg.imie_wroga,
  COUNT(*)
FROM
  (select * from incydenty order by imie_wroga) i LEFT JOIN wrogowie_ref wg ON i.imie_wroga = wg.imie_wroga
WHERE stopien = 7
GROUP BY
  i.pseudo, wg.imie_wroga

26 rows affected.

QUERY PLAN
HashAggregate (cost=238485.00..238485.20 rows=20 width=21) (actual time=862.024..862.026 rows=5.00 loops=1)
"Group Key: incydenty.pseudo, wg.imie_wroga"
Batches: 1 Memory Usage: 32kB
"Buffers: local read=9409, temp read=2276 written=2913"
-> Merge Join (cost=221610.00..237860.00 rows=83333 width=13) (actual time=820.312..845.910 rows=83333.00 loops=1)
Merge Cond: (incydenty.imie_wroga = wg.imie_wroga)
"Buffers: local read=9409, temp read=2276 written=2913"
-> Sort (cost=221608.84..224108.84 rows=1000000 width=89) (actual time=681.671..776.438 rows=750002.00 loops=1)
Sort Key: incydenty.imie_wroga
Sort Method: external merge Disk: 23192kB


## Jednostka kosztu: z czego składa się liczba w planach

Koszt prezentowany w planie wykonania nie jest wyrażony w milisekundach — jest to **suma umownych jednostek**, których wagi są zdefiniowane w parametrach konfiguracyjnych planera (można je podejrzeć oraz modyfikować):

- `seq_page_cost` (= `1.0` domyślnie) — koszt odczytu pojedynczej strony danych **sekwencyjnie** (pełny skan tabeli);
- `random_page_cost` (= `4.0` domyślnie) — koszt losowego odczytu strony danych (np. podczas czytania po indeksie);
- `cpu_tuple_cost` (= `0.01` domyślnie) — koszt przetworzenia pojedynczego wiersza przez operator.

Koszt pełnego skanu tabeli można zatem wyznaczyć ze wzoru:

$$	ext{koszt} = 	ext{liczba stron} 	imes 	ext{seq\_page\_cost} + 	ext{liczba wierszy} 	imes 	ext{cpu\_tuple\_cost}$$

W analizowanym przypadku tabela `incydenty` zajmuje 9408 stron przy milionie wierszy, co daje:

$$9408 	imes 1{,}0 + 1\,000\,000 	imes 0{,}01 = 19\,408	ext{ jednostek}$$

Dokładnie tę wartość można zaobserwować jako koszt całkowity operatora `Seq Scan` w planie (natomiast liczbę stron potwierdzają sekcja `BUFFERS` oraz widok katalogowy `pg_class`). Model kosztu odnosi się do **liczby stron i wierszy, a nie do czasu zegarowego**: nawet na wolniejszym nośniku danych wyliczony koszt nadal wyniesie `19 408`. Z tego względu koszty należy porównywać **wyłącznie względem siebie** (plan A kontra plan B w ramach tego samego silnika bazodanowego), natomiast rzeczywistą szybkość wykonania pozwalają ocenić dopiero pomiary.

Modyfikacja wag parametrów wpływa bezpośrednio na decyzje optymalizatora. Przykładowo zwiększenie wartości `random_page_cost` skłania planer do częstszego wybierania pełnych skanów sekwencyjnych, natomiast jej zmniejszenie faworyzuje operacje odczytu przez indeksy (których na obecnych zajęciach nie tworzymy).

In [14]:
%%sql pgkotki
-- Rozliczenie jednostki kosztu na sztukach:
SHOW seq_page_cost;

1 rows affected.

seq_page_cost
1


In [13]:
%%sql pgkotki
SHOW cpu_tuple_cost

1 rows affected.

cpu_tuple_cost
0.01


In [15]:
%%sql pgkotki
-- Strony i wiersze z katalogu (statystyka po ANALYZE z setupu).
-- Przycinamy do schematu tabel tymczasowych TEJ sesji (pg_my_temp_schema):
-- bez tego pg_class naliczyłby też "incydenty" po poprzednich sesjach zajęć.
SELECT relpages AS strony_tabeli, reltuples::bigint AS wierszy
FROM pg_class
WHERE relname = 'incydenty'
  AND relnamespace = pg_my_temp_schema()


1 rows affected.

strony_tabeli,wierszy
9408,1000000


In [37]:
%%sql pgkotki
-- Porównaj koszt tego Seq Scanu z powyższą arytmetyką:
EXPLAIN
SELECT
  id
FROM
  incydenty

1 rows affected.

QUERY PLAN
Seq Scan on incydenty (cost=0.00..19408.00 rows=1000000 width=4)


In [41]:
%%sql pgkotki
ANALYZE incydenty

++
||
++
++

In [44]:
%%sql pgkotki
-- Estymata kontra rzeczywistość: dwa filtry, dwa światy.
-- (a) kolumna kraj MA statystyki (po ANALYZE):
EXPLAIN ANALYZE
SELECT
  id
FROM
  incydenty
WHERE
  kraj = 'US'

8 rows affected.

QUERY PLAN
Seq Scan on incydenty (cost=0.00..21908.00 rows=144433 width=4) (actual time=0.044..137.694 rows=142857.00 loops=1)
Filter: (kraj = 'US'::text)
Rows Removed by Filter: 857143
Buffers: local read=9408
Planning:
Buffers: shared hit=3
Planning Time: 0.059 ms
Execution Time: 141.209 ms


In [43]:
%%sql pgkotki
-- (b) wyrażenie lower(pseudo) — DLA PLANERA STATYSTYK NIE MA:
EXPLAIN ANALYZE
SELECT
  id
FROM
  incydenty
WHERE
  LOWER(pseudo) = 'kot_1'

8 rows affected.

QUERY PLAN
Seq Scan on incydenty (cost=0.00..24408.00 rows=5000 width=4) (actual time=0.047..257.491 rows=50000.00 loops=1)
Filter: (lower(pseudo) = 'kot_1'::text)
Rows Removed by Filter: 950000
Buffers: local read=9408
Planning:
Buffers: shared hit=3
Planning Time: 0.068 ms
Execution Time: 258.999 ms


**Porównaj szacowaną liczbę wierszy (`rows`) z wartością rzeczywistą (`actual rows`).**

Z czego wynika ta rozbieżność?
- Dla kolumny `kraj` planer dysponuje w widoku `pg_stats` histogramem oraz listą najczęstszych wartości — wie zatem, że wartość `'US'` stanowi około 1/7 zbioru.
- Dla wyrażenia `LOWER(pseudo)` statystyki nie zostały zgromadzone, więc optymalizator stosuje domyślną selektywność dla predykatu równościowego (ułamek procenta wierszy).

Błąd estymacji o rząd wielkości może wydawać się niegroźny, dopóki nie zostanie propagowany **w górę drzewa wykonania**. Należy pamiętać, że na podstawie tych szacunków optymalizator przydziela budżet pamięci dla sortowania, wybiera algorytm złączenia oraz wyznacza liczbę partycji dla operacji haszowania.

**Wniosek praktyczny:** porównanie pary `rows` i `actual rows` w każdym węźle planu stanowi podstawowy mechanizm wczesnego wykrywania błędów optymalizatora.

### Na co zwracać uwagę w planie wykonania

1. **`rows` różni się od `actual rows` o rząd wielkości** — wskazuje na brakujące lub nieaktualne statystyki bądź niestatystykowalne wyrażenia (funkcje na kolumnach, rzutowania typów, `LOWER`/`UPPER` — jak w powyższym przykładzie).
2. **`Rows Removed by Filter` przyjmuje duże wartości** — operator filtrowania odczytuje miliony wierszy, aby zwrócić niewielki ułamek zbioru; w takich sytuacjach warto rozważyć utworzenie indeksu lub przebudowę zapytania.
3. **`actual time × loops`** — nawet jeśli pojedyncze wykonanie węzła jest bardzo szybkie, jego wielokrotne wywołanie (np. po stronie wewnętrznej `Nested Loop` dla dużej liczby rekordów zewnętrznych) generuje wysoki łączny czas pracy; zawsze należy analizować ten iloczyn.
4. **Wysoka wartość `width` tam, gdzie nie jest to konieczne** — przekazywanie zbędnych, szerokich kolumn między operatorami (np. wskutek użycia `SELECT *` przed złączeniem lub sortowaniem).
5. **Przejście operatorów `Sort` lub `Hash` na dysk** — pojawienie się w planie wpisów `external merge`, `Batches > 1` lub wzrost wartości `Disk Usage`.

## Statystyki są w katalogu — można je podejrzeć

Planer nie przechowuje w pamięci podręcznej informacji o zawartości tabel z poprzednich zapytań — całą wiedzę o rozkładzie danych czerpie z metadanych zapisanych w **katalogu systemowym**.

Dwa kluczowe obiekty katalogowe wykorzystywane przez optymalizator w PostgreSQL to:

- `pg_class` — zawiera jeden wiersz na relację: m.in. `reltuples` (szacowana liczba wierszy) oraz `relpages` (liczba stron dyskowych). Wartości te są aktualizowane przez polecenie `ANALYZE`; po masowych operacjach `INSERT`/`UPDATE`/`DELETE` pozostają one **nieaktualne do momentu ponownego uruchomienia `ANALYZE` lub procesu `autovacuum`**.
- `pg_stats` — przechowuje statystyki **dla poszczególnych kolumn**: histogramy wartości, listy najczęściej występujących wartości (`most_common_vals`, `most_common_freqs`), odsetek wartości `NULL` (`null_frac`) oraz średnią szerokość danych.

Sprawdźmy, jak prezentują się te metadane dla kolumn `kraj` (7 unikalnych wartości) oraz `szkodliwosc` (1000 unikalnych wartości, `NULL` występujący co ~97 wierszy). Dla predykatu `kraj = 'US'` planer odwoła się do tablicy najczęstszych wartości, natomiast dla warunku `szkodliwosc IS NULL` wykorzysta współczynnik `null_frac` — warto porównać te estymaty z rzeczywistą liczebnością wierszy.

In [49]:
%%sql pgkotki
-- Katalog mówi planerowi, ile czego jest (i w jakich proporcjach).
-- Ile wierszy i stron "widzi" planer (uzupełni ANALYZE z setupu) —
-- tylko tabele tymczasowe bieżącej sesji:
SELECT relname, reltuples::bigint AS wierszy, relpages AS strony
FROM pg_class
WHERE relnamespace = pg_my_temp_schema() AND relkind = 'r'
ORDER BY relname


2 rows affected.

relname,wierszy,strony
incydenty,1000000,9408
wrogowie_ref,12,1


In [50]:
%%sql pgkotki
-- Statystyki kolumny kraj: ile wartości, które najczęstsze:
SELECT
  attname,
  n_distinct,
  null_frac,
  most_common_vals,
  most_common_freqs
FROM
  pg_stats
WHERE
  schemaname = (
    SELECT
      nspname
    FROM
      pg_namespace
    WHERE
      oid = pg_my_temp_schema ()
  )
  AND attname IN ('kraj', 'pseudo')

2 rows affected.

attname,n_distinct,null_frac,most_common_vals,most_common_freqs
kraj,7.0,0.0,"{DE,US,FR,IT,UK,PL,ES}","[0.14616667, 0.14443333, 0.14343333, 0.14213334, 0.14173333, 0.1415, 0.1406]"
pseudo,20.0,0.0,"{KOT_0,KOT_2,KOT_7,KOT_3,KOT_5,KOT_19,KOT_13,KOT_8,KOT_18,KOT_9,KOT_17,KOT_1,KOT_11,KOT_10,KOT_12,KOT_6,KOT_14,KOT_16,KOT_4,KOT_15}","[0.051466666, 0.05136667, 0.05136667, 0.051333334, 0.051, 0.0509, 0.050866667, 0.050466668, 0.0503, 0.050066665, 0.050033335, 0.049733333, 0.049666665, 0.0493, 0.04916667, 0.0488, 0.0487, 0.0487, 0.04853333, 0.048233334]"


In [51]:
%%sql pgkotki
-- Planer wie o NULL-ach w szkodliwosci — sprawdź proporcje:
SELECT
  attname,
  null_frac,
  n_distinct
FROM
  pg_stats
WHERE
  schemaname = (
    SELECT
      nspname
    FROM
      pg_namespace
    WHERE
      oid = pg_my_temp_schema ()
  )
  AND attname = 'szkodliwosc'

1 rows affected.

attname,null_frac,n_distinct
szkodliwosc,0.009733333,1000.0


In [52]:
%%sql pgkotki
-- ...a rzeczywistość (policz ręcznie i porównaj):
SELECT
  COUNT(*) FILTER (
    WHERE
      kraj = 'US'
  ) AS us_razy,
  COUNT(*) FILTER (
    WHERE
      szkodliwosc IS NULL
  ) AS nullow
FROM
  incydenty

1 rows affected.

us_razy,nullow
142857,10309


## Trzy dialekty czytania planu

Plany wykonania są dostępne we wszystkich trzech analizowanych silnikach bazodanowych. **Oracle i SQL Server stosują odmienną składnię oraz sposób prezentacji**, jednak podstawowy **słownik operatorów fizycznych pozostaje spójny** (np. złączenie haszowe wszędzie występuje jako `Hash Join` / `Hash Match`). Szczegółowe zestawienie wygląda następująco:

- **PostgreSQL:** `EXPLAIN` / `EXPLAIN ANALYZE` — tekstowe drzewo operatorów zwracane jako wynik zapytania; wartości szacowane i rzeczywiste prezentowane są w pojedynczym widoku.
- **Oracle:** procedura trzyetapowa — `EXPLAIN PLAN FOR` zapisuje plan do tabeli `PLAN_TABLE`, a funkcja `DBMS_XPLAN.DISPLAY` renderuje drzewo w formie tekstowej. Jest to plan **szacunkowy** (odpowiednik `EXPLAIN` bez `ANALYZE`); rzeczywiste statystyki wykonania można odczytać za pomocą `DBMS_XPLAN.DISPLAY_CURSOR`, co wymaga jednak uprawnień do dynamicznych widoków wydajnościowych `V$`.
- **SQL Server:** umożliwia wyświetlenie planu graficznego w środowisku SSMS (skrót `Ctrl+M`), jednak samo **prawo do generowania planów** (`SHOWPLAN`) jest odrębnym uprawnieniem bazodanowym; konto laboratoryjne nie posiada tego uprawnienia w bazie Chinook (co za chwilę zweryfikujemy na przykładzie komunikatu błędu `Msg 262`). Plan wykonania z rzeczywistymi metrykami można natomiast pobrać dla tabel tymczasowych za pomocą opcji `STATISTICS PROFILE` lub `STATISTICS XML`.

Jednostki kosztu również różnią się definicją: w PostgreSQL bazą jest odczyt strony sekwencyjnej, w Oracle — pojedyncza operacja *single-block I/O*, a w SQL Serverze — wewnętrzna jednostka estymacji. Dlatego wartości kosztu należy porównywać wyłącznie **w obrębie tego samego silnika i tego samego zapytania**.

W kolejnych przykładach przeanalizujemy plany wykonania w Oracle (na bazie Chinook: złączenie `INVOICELINE × TRACK` po `TRACKID` z użyciem wskazówek `/*+ ... */`) oraz w SQL Serverze (na tabeli tymczasowej `#incydenty` z wymuszaniem algorytmów klauzulą `OPTION`).

In [89]:
%%sql oracle
select * from c##Kotki.Kocury;


imie,plec,pseudo,funkcja,szef,w_stadku_od,przydzial_myszy,myszy_extra,nr_bandy
MRUCZEK,M,TYGRYS,SZEFUNIO,None,2002-01-01 00:00:00,103,33,1
CHYTRY,M,BOLEK,DZIELCZY,TYGRYS,2002-05-05 00:00:00,50,None,1
KOREK,M,ZOMBI,BANDZIOR,TYGRYS,2004-03-16 00:00:00,75,13,3
BOLEK,M,LYSY,BANDZIOR,TYGRYS,2006-08-15 00:00:00,72,21,2
ZUZIA,D,SZYBKA,LOWCZY,LYSY,2006-07-21 00:00:00,65,None,2
RUDA,D,MALA,MILUSIA,TYGRYS,2006-09-17 00:00:00,22,42,1
PUCEK,M,RAFA,LOWCZY,TYGRYS,2006-10-15 00:00:00,65,None,4
PUNIA,D,KURKA,LOWCZY,ZOMBI,2008-01-01 00:00:00,61,None,3
BELA,D,LASKA,MILUSIA,LYSY,2008-02-01 00:00:00,24,28,2
KSAWERY,M,MAN,LAPACZ,RAFA,2008-07-12 00:00:00,51,None,4


In [91]:
%%sql oracle
-- Oracle: EXPLAIN PLAN zapisuje plan w PLAN_TABLE (współdzielona globalnie!),
-- więc używamy STATEMENT_ID, żeby nie zbiegały się eksperymenty.
-- DBMS_XPLAN.DISPLAY z parametrami NAZWANYMI renderuje drzewo.
DELETE FROM plan_table
WHERE statement_id = 'cw4_sel';

EXPLAIN PLAN
SET STATEMENT_ID = 'cw4_sel' FOR
SELECT
  COUNT(*) AS licznik
FROM
  c##Kotki.Kocury;

SELECT *
FROM TABLE (
  DBMS_XPLAN.DISPLAY (
    table_name   => 'PLAN_TABLE',
    statement_id => 'cw4_sel',
    format       => 'ALL'
  )
);

3 rows affected.

plan_table_output
Plan hash value: 4234372488
""
---------------------------------------------------------------------
| Id | Operation | Name | Rows | Cost (%CPU)| Time |
---------------------------------------------------------------------
| 0 | SELECT STATEMENT | | 1 | 1 (0)| 00:00:01 |
| 1 | SORT AGGREGATE | | 1 | | |
| 2 | INDEX FULL SCAN| KO_PS_PK | 18 | 1 (0)| 00:00:01 |
---------------------------------------------------------------------
""


## SQL Server

W SQL Serverze samo **wyświetlenie planu wykonania** wymaga posiadania uprawnienia `SHOWPLAN` — jego brak skutkuje błędem **Msg 262** (*„SHOWPLAN permission denied in database `XXX`”*).

Generowanie szacowanych planów można włączyć za pomocą instrukcji `SET SHOWPLAN_ALL`, `SET SHOWPLAN_TEXT` oraz `SET SHOWPLAN_XML`.

Gdy opcja `SET SHOWPLAN_ALL` jest ustawiona na `ON`, SQL Server zwraca informacje o planowanym sposobie wykonania każdej instrukcji bez jej faktycznego uruchamiania. Po aktywacji tej opcji zwracane są wyłącznie plany dla kolejnych zapytań Transact-SQL, aż do momentu jej wyłączenia za pomocą `SET SHOWPLAN_ALL OFF` (szczegóły w [dokumentacji Microsoft Learn](https://learn.microsoft.com/en-us/sql/t-sql/statements/set-showplan-all-transact-sql?view=sql-server-ver17)).

In [44]:
%%sql mssql
SET SHOWPLAN_TEXT OFF;

++
||
++
++

In [4]:
%%sql mssql
DROP TABLE IF EXISTS #incydenty;

CREATE TABLE #incydenty (
        id INT IDENTITY PRIMARY KEY,
        pseudo NVARCHAR(20), imie_wroga NVARCHAR(20),
        data_incydentu DATE, szkodliwosc INT, kraj NVARCHAR(2))

INSERT INTO #incydenty (pseudo, imie_wroga, data_incydentu, szkodliwosc, kraj)
    SELECT TOP (300000)
           'KOT_' + CAST(g % 20 AS NVARCHAR(20)),
           'WROG_' + CAST(g % 12 AS NVARCHAR(20)),
           DATEADD(DAY, g % 3650, '2015-01-01'),
           CASE WHEN g % 97 = 0 THEN NULL ELSE g % 1000 END,
           CASE WHEN g % 7 = 0 THEN 'US' ELSE 'PL' END
    FROM (SELECT ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS g
          FROM sys.all_columns a CROSS JOIN sys.all_columns b) x

SELECT COUNT(*) FROM #incydenty

Switching to connection 'mssql'

""
300000


In [113]:
%%sql mssql
SET SHOWPLAN_ALL ON

++
||
++
++

In [105]:
%%sql mssql
    SELECT COUNT(*) FROM Kotki.dbo.Kocury;

""
18


Plan wykonania wygenerowany z opcją `STATISTICS PROFILE` odczytuje się w formie drzewa tekstowego: wcięcia oznaczone symbolem `|--` wyznaczają kolejne poziomy hierarchii (od korzenia w dół), a dwie kluczowe metryki dla każdego węzła to **`Rows`** (rzeczywista liczba wierszy — odpowiednik `actual rows` w PostgreSQL) oraz **`EstimateRows`** (liczba szacowana — odpowiednik `rows` w PostgreSQL).

Poniżej przedstawiono plan dla prostego zliczania wierszy — warto porównać go z analogicznym `EXPLAIN ANALYZE SELECT COUNT(*) FROM incydenty` w PostgreSQL: **identyczna struktura drzewa, lecz inne nazwy operatorów** (`Stream Aggregate` / `Compute Scalar` zamiast `Aggregate`; `Clustered Index Scan` zamiast `Seq Scan`, ponieważ tabela tymczasowa z kluczem głównym `PRIMARY KEY` jest fizycznie zorganizowana jako indeks klastrowany).

In [114]:
%%sql mssql 
SELECT COUNT(*)
FROM #incydenty

StmtText,StmtId,NodeId,Parent,PhysicalOp,LogicalOp,Argument,DefinedValues,EstimateRows,EstimateIO,EstimateCPU,AvgRowSize,TotalSubtreeCost,OutputList,Warnings,Type,Parallel,EstimateExecutions
SELECT COUNT(*)FROM #incydenty,1,1,0,None,None,1,None,1.0,None,None,None,2.034764051437378,None,None,SELECT,False,None
"|--Compute Scalar(DEFINE:([Expr1002]=CONVERT_IMPLICIT(int,[Expr1003],0)))",1,2,1,Compute Scalar,Compute Scalar,"DEFINE:([Expr1002]=CONVERT_IMPLICIT(int,[Expr1003],0))","[Expr1002]=CONVERT_IMPLICIT(int,[Expr1003],0)",1.0,0.0,0.0,11,2.034764051437378,[Expr1002],None,PLAN_ROW,False,1.0
|--Stream Aggregate(DEFINE:([Expr1003]=Count(*))),1,3,2,Stream Aggregate,Aggregate,None,[Expr1003]=Count(*),1.0,0.0,0.18000049889087677,11,2.034764051437378,[Expr1003],None,PLAN_ROW,False,1.0
|--Clustered Index Scan(OBJECT:([tempdb].[dbo].[#incydenty])),1,4,3,Clustered Index Scan,Clustered Index Scan,OBJECT:([tempdb].[dbo].[#incydenty]),None,300000.0,1.524606466293335,0.33015701174736023,9,1.8547635078430176,None,None,PLAN_ROW,False,1.0


In [111]:
%%sql mssql 
# Plan wykonywany z STATISTICS PROFILE — helper na surowym pymssql:
# (plan przyjezdza w DRUGIM zbiorze wynikow, wiec %%sql tego nie pokaze - musimy sięgnąc po rozwiązanie alternatywne i ręcznie wywołać polecenie)
SELECT COUNT(*) AS ile
FROM #incydenty i
JOIN (SELECT 'WROG_' + CAST(n AS NVARCHAR(20)) AS imie_wroga
      FROM (VALUES (0),(1),(2),(3),(4),(5),(6),(7),(8),(9),(10),(11)) v(n)) w
      ON w.imie_wroga = i.imie_wroga

ile
300000


In [130]:
from sql.connection import ConnectionManager

def get_dbapi_connection(alias=None):
    jconn = ConnectionManager.connections[alias] if alias else ConnectionManager.current

    # Names under which different JupySQL versions keep the SQLAlchemy Connection
    for name in ("connection_sqlalchemy", "_connection", "session", "_session"):
        sa_conn = getattr(jconn, name, None)
        if sa_conn is not None and hasattr(sa_conn, "connection"):
            return sa_conn.connection          # SQLAlchemy Connection -> DBAPI connection proxy

    # Fallback: the wrapper itself might be usable
    if hasattr(jconn, "cursor"):
        return jconn

    raise AttributeError(
        "Could not find the SQLAlchemy connection. Available attributes: "
        + ", ".join(a for a in dir(jconn) if not a.startswith("__"))
    )

def profile(sql, alias=None):
    raw = get_dbapi_connection(alias)          # DBAPI connection
    cur = raw.cursor()
    try:
        cur.execute("SET NOCOUNT ON; SET STATISTICS PROFILE ON;")
        cur.execute(sql)

        # 1st result set: the query's own rows
        cols = [c[0] for c in cur.description]
        rows = [dict(zip(cols, r)) for r in cur.fetchall()]

        # 2nd result set: the profile
        plan = []
        if cur.nextset():
            pcols = [c[0] for c in cur.description]
            plan = [dict(zip(pcols, r)) for r in cur.fetchall()]
        return rows, plan
    finally:
        cur.execute("SET STATISTICS PROFILE OFF")
        cur.close()

def show(rows, columns, width=40):
    """Print a list of dicts as a simple text table."""
    cell = lambda v: "" if v is None else str(v)[:width]
    w = [max(len(c), *(len(cell(r[c])) for r in rows)) if rows else len(c) for c in columns]
    line = lambda vals: "  ".join(v.ljust(n) for v, n in zip(vals, w))
    print(line(columns))
    print(line(["-" * n for n in w]))
    for r in rows:
        print(line([cell(r[c]) for c in columns]))

def print_tree(plan):
    """Print a tree like view of the plan."""
    children = {}
    for r in plan:
        children.setdefault(r["Parent"], []).append(r)

    def walk(node, depth):
        print(f'{"  " * depth}{node["PhysicalOp"] or node["StmtText"].strip()}'
              f'  (actual={node["Rows"]}, est={node["EstimateRows"]})')
        for ch in children.get(node["NodeId"], []):
            walk(ch, depth + 1)

    for root in children.get(0, []):           # the root has Parent = 0
        walk(root, 0)

rows, plan = profile("""
SELECT pseudo
FROM Kotki.dbo.Kocury 
""", "mssql")

#show(plan, ["Rows", "Executes", "EstimateRows", "PhysicalOp", "LogicalOp"])
print_tree(plan)

SELECT pseudo
FROM Kotki.dbo.Kocury  (actual=18, est=18.0)
  Clustered Index Scan  (actual=18, est=18.0)


In [115]:
%%sql mssql    
SET SHOWPLAN_ALL OFF

++
||
++
++

## Tabela podsumowująca: ten sam plan w trzech dialektach

| Cecha | PostgreSQL | Oracle | SQL Server |
|---|---|---|---|
| Szacunek planu | `EXPLAIN` | `EXPLAIN PLAN` → `DBMS_XPLAN.DISPLAY` | `SET SHOWPLAN_ALL ON` (wymaga uprawnienia) |
| Plan + rzeczywistość | `EXPLAIN ANALYZE` | `DBMS_XPLAN.DISPLAY_CURSOR` (wymaga dostępu do widoków `V$`) | `SET STATISTICS PROFILE/XML ON` (drugi zbiór wyników) |
| Rzeczywiste wiersze per węzeł | `actual rows` | `A-Rows` | `Rows` |
| Szacunek wierszy | `rows` | `E-Rows` | `EstimateRows` |
| Koszt | `cost=` (strona seq. = `1.0`) | `Cost` (*single-block I/O*) | `TotalSubtreeCost` (skala wewnętrzna) |
| Pełny przegląd tabeli | `Seq Scan` | `TABLE ACCESS FULL` | `Clustered Index Scan` / `Table Scan` |
| Agregacja | `Aggregate` / `HashAggregate` | `SORT AGGREGATE` / `HASH GROUP BY` | `Stream Aggregate` / `Hash Match (Aggregate)` |
| Sortowanie | `Sort` | `SORT ORDER BY` | `Sort` |

Należy pamiętać, że koszt jest porównywalny wyłącznie między alternatywnymi planami dla tego samego zapytania w obrębie jednego silnika. Koszt danego węzła w planie uwzględnia sumaryczne koszty jego węzłów podrzędnych, dlatego porównuje się całkowite koszty w korzeniu drzewa.

Właściwym zastosowaniem metryki kosztu jest **porównanie w obrębie jednego zapytania**: zestawienie kosztu `Seq Scan` (`19 408`) z kosztem `Hash Join` (`33 159`) w tym samym planie pozwala ocenić, którą operację optymalizator wycenił jako bardziej obciążającą. Porównywanie wartości kosztu **między różnymi silnikami bazodanowymi** jest bezcelowe.

Należy jednak uwzględnić **pułapkę modelu kosztowego**: niższy koszt teoretyczny oznacza plan preferowany przez optymalizator, ale nie zawsze gwarantuje krótszy czas wykonania w milisekundach. Na rzeczywistych danych model kosztu może faworyzować inne rozwiązanie niż wskazują pomiary czasu — dlatego w diagnostyce należy **analizować zarówno koszt, jak i rzeczywisty czas wykonania**.

## Podsumowanie bloku

1. **Plan wykonania = drzewo operatorów** odczytywane od liści (źródeł danych) do korzenia (wyniku końcowego). Zapytanie SQL określa *co* uzyskać, natomiast plan opisuje *jak* to zrealizować.
2. **Statystyki katalogowe** (`reltuples`, `pg_stats`) stanowią jedyne źródło wiedzy planera o rozkładzie danych — bez aktualizacji `ANALYZE` optymalizator opiera się na wartościach domyślnych (podobnie jak w przypadku wyrażeń typu `LOWER(kolumna)`).
3. **Para `rows` / `actual rows`** stanowi podstawowy wskaźnik poprawności estymacji: rozbieżność rzędu wielkości oznacza, że wyższe węzły planu zostały zbudowane na błędnych założeniach.
4. **Koszt nie jest tożsamy z czasem** — umowne jednostki (oparte na operacjach I/O i CPU) służą do porównywania planów w obrębie jednego silnika, podczas gdy rzeczywisty czas mierzy `EXPLAIN ANALYZE` (PostgreSQL) lub `STATISTICS PROFILE/TIME` (SQL Server). Nie zawsze zachodzi prosta zależność „niższy koszt = szybsze wykonanie”.
5. **Trzy silniki, trzy zestawy narzędzi**: PostgreSQL (`EXPLAIN (ANALYZE, BUFFERS)`), Oracle (`EXPLAIN PLAN` + `DBMS_XPLAN`) oraz SQL Server (`SHOWPLAN` / `STATISTICS PROFILE`) realizują tę samą koncepcję za pomocą odmiennych interfejsów, przy czym dostęp do nich zależy od posiadanych uprawnień (np. błąd `Msg 262`).

**Co dalej:** znając strukturę planów wykonania, przeanalizujemy kluczową decyzję optymalizatora — **wybór algorytmu złączenia**: *Nested Loops*, *Merge Join* oraz *Hash Join*. Na zbiorze miliona incydentów różnice między nimi będą wyraźnie widoczne zarówno w kosztach, jak i w czasie wykonania.

# Złączenia: trzy algorytmy, jedna decyzja

**Pytanie bloku:** gdy dwa zbiory danych spotykają się w operacji `JOIN`, optymalizator dysponuje trzema podstawowymi algorytmami złączeń — którym z nich się posłuży, dlaczego podejmuje taką decyzję i jakie są konsekwencje ręcznego wymuszenia suboptymalnego algorytmu?

## Trzy algorytmy, jedna decyzja

Operacja `JOIN` posiada kilka odmiennych **fizycznych algorytmów realizacji**. W niniejszym bloku skupimy się na trzech podstawowych mechanizmach:

1. **Nested Loops (NL)** — dla każdego wiersza ze zbioru zewnętrznego (lewego) przeszukiwane są pasujące wiersze ze zbioru wewnętrznego (prawego). Złożoność bez indeksu rośnie jak $|L| \cdot |R|$. W naszym laboratorium lewa strona liczy milion wierszy, a prawa 12 — zatem nawet przy użyciu operatora `Materialize` (buforującego mniejszą tabelę w pamięci podręcznej) silnik wykonawczy musi przeprowadzić ok. 12 milionów porównań. Algorytm ten jest niezwykle efektywny, gdy zbiór zewnętrzny jest **mały**, a na kluczu złączenia zbioru wewnętrznego istnieje **indeks** — ponieważ na obecnych zajęciach nie tworzymy indeksów, zachowanie NL zaobserwujemy poprzez jego ręczne wymuszenie.
2. **Merge Join** — wymaga, aby oba zbiory wejściowe były posortowane według klucza złączenia, po czym wykonuje pojedyncze scalenie za pomocą równoległych wskaźników. Złożoność samego scalania wynosi $pprox |L| + |R|$, jednak należy doliczyć koszt wcześniejszego sortowania obu stron, jeśli nie są one uporządkowane (w naszym przypadku brak indeksów sprawia, że pod operatorem `Merge Join` zawsze znajdują się operatory `Sort`).
3. **Hash Join** — mniejszy zbiór danych jest ładowany do tablicy haszującej w pamięci operacyjnej (faza *build*), a następnie większy zbiór jest jednokrotnie skanowany w poszukiwaniu dopasowań (faza *probe*). Złożoność wynosi $pprox |L| + |R|$ **bez konieczności sortowania** — jest to domyślny wybór dla dużych złączeń bez indeksów.

Ogólna heurystyka wyboru algorytmu:
- mały zbiór × duży zbiór z indeksem $ightarrow$ `Nested Loops`;
- wejścia już posortowane (np. z indeksów) $ightarrow$ `Merge Join`;
- duże, nieposortowane zbiory (zwłaszcza duża × duża) $ightarrow$ `Hash Join`.

Nie jest to jednak sztywna reguła — ostateczną decyzję podejmuje optymalizator na podstawie **szacunków kardynalności** i modelu kosztowego, choć programista może wpływać na ten wybór za pomocą parametrów sesyjnych lub wskazówek (hintów). Dodatkowo każdy z algorytmów ma inne zapotrzebowanie na pamięć operacyjną: `Nested Loops` nie wymaga bufora na dane pośrednie, natomiast `Merge Join` oraz `Hash Join` potrzebują przestrzeni roboczej.

### Co wybiera domyślnie? (milion × dwanaście)

Rozważmy zapytanie zliczające liczbę incydentów w podziale na stopień zagrożenia wroga.

Mamy tu do czynienia ze złączeniem typu **duża × mała**: milion wierszy z tabeli `incydenty` (strona lewa) łączonych z 12 rekordami ze słownika `wrogowie_ref` (strona prawa).

Statystyki katalogowe informują planer, że prawa relacja zawiera 12 wierszy, a lewa — milion. Szacowany nakład pracy wynosi więc: dla `Nested Loops` — $10^6$ iteracji po 12 elementów; dla `Merge Join` — koszt pełnego sortowania miliona wierszy; dla `Hash Join` — jednokrotny skan dużej tabeli oraz budowa niewielkiej tablicy haszującej z 12 wpisów. Model kosztowy jednoznacznie wskazuje na `Hash Join`.

W planie wykonania pojawią się trzy charakterystyczne węzły: pełny skan sekwencyjny dużej tabeli (`Seq Scan`), węzeł `Hash` (jednorazowa budowa tablicy haszującej ze słownika wrogów wraz z parametrami `Buckets`, `Batches` i `Memory Usage`) oraz sam operator `Hash Join`. Warto zwrócić uwagę na minimalne zużycie pamięci (`Memory Usage: 9kB`).

In [131]:
%%sql pgkotki
-- Domyślny wybór planera dla zlaczenia 1M x 12:
EXPLAIN ANALYZE
SELECT
  w.stopien,
  COUNT(*) AS incydentow
FROM
  incydenty i
  JOIN wrogowie_ref w ON w.imie_wroga = i.imie_wroga
GROUP BY
  w.stopien
ORDER BY
  w.stopien

Switching to connection 'pg'

22 rows affected.

QUERY PLAN
Sort (cost=38159.54..38159.56 rows=10 width=12) (actual time=372.667..372.669 rows=10.00 loops=1)
Sort Key: w.stopien
Sort Method: quicksort Memory: 25kB
Buffers: local read=9409
-> HashAggregate (cost=38159.27..38159.37 rows=10 width=12) (actual time=372.659..372.662 rows=10.00 loops=1)
Group Key: w.stopien
Batches: 1 Memory Usage: 32kB
Buffers: local read=9409
-> Hash Join (cost=1.27..33159.27 rows=1000000 width=4) (actual time=0.073..259.790 rows=1000000.00 loops=1)
Hash Cond: (i.imie_wroga = w.imie_wroga)


### Zabraniamy haszowania: co zostaje?

PostgreSQL umożliwia **wyłączanie wybranych algorytmów planera za pomocą parametrów sesyjnych** `enable_hashjoin`, `enable_mergejoin` oraz `enable_nestloop` (technicznie nie jest to bezwzględny zakaz, lecz dodanie wysokiej kary kosztowej rzędu $+10^{10}$ — planer nadal może użyć danego operatora, jeśli nie istnieje żadna alternatywa, np. dla złączeń bez warunku równościowego).

Po ustawieniu `enable_hashjoin = off` optymalizator ma do wyboru `Merge Join` oraz `Nested Loops`. Przy połączeniu miliona incydentów z 12 wrogami algorytm `Merge Join` wymaga **uprzedniego posortowania obu stron** — wymusza to wstawienie operatora `Sort` nad pełnym skanem tabeli `incydenty` (sortowanie zewnętrzne *external merge*, ponieważ dane przekraczają domyślny budżet pamięci roboczej).

**Przewidywanie:** w planie pojawią się dwa węzły sortowania — po jednym dla każdego wejścia złączenia scalającego. Mniejszy zbiór (12 wierszy) zmieści się w pamięci operacyjnej (`quicksort, Memory: 25kB`), natomiast większy zostanie posortowany z użyciem plików tymczasowych na dysku (`external merge`).

In [132]:
%%sql pgkotki
-- Bez haszowania: planer zostaje z Merge (i NL jako rezerwa).
SET
  enable_hashjoin = off;

EXPLAIN ANALYZE
SELECT
  w.stopien,
  COUNT(*) AS incydentow
FROM
  incydenty i
  JOIN wrogowie_ref w ON w.imie_wroga = i.imie_wroga
GROUP BY
  w.stopien
ORDER BY
  w.stopien;

28 rows affected.

QUERY PLAN
Sort (cost=155242.00..155242.03 rows=10 width=12) (actual time=1033.192..1033.194 rows=10.00 loops=1)
Sort Key: w.stopien
Sort Method: quicksort Memory: 25kB
"Buffers: local read=9409, temp read=1368 written=1373"
-> HashAggregate (cost=155241.74..155241.84 rows=10 width=12) (actual time=1033.184..1033.187 rows=10.00 loops=1)
Group Key: w.stopien
Batches: 1 Memory Usage: 32kB
"Buffers: local read=9409, temp read=1368 written=1373"
-> Merge Join (cost=132741.68..150241.74 rows=1000000 width=4) (actual time=581.749..923.171 rows=1000000.00 loops=1)
Merge Cond: (w.imie_wroga = i.imie_wroga)


Porównajmy uzyskany plan z poprzednim wariantem (`Hash Join`):

- Szczyt złączenia stanowi teraz **`Merge Join`**, a pod nim znajdują się **dwa operatory `Sort`** — jest to narzut algorytmu scalającego wynikający z konieczności uporządkowania obu wejść. Sortowanie dużej tabeli raportuje `Sort Method: external merge Disk: 10944kB`, co oznacza przekroczenie dostępnej pamięci roboczej (parametru `work_mem`, szerzej omówionego w kolejnym rozdziale).
- **Brak tablicy haszującej nie oznacza szybszego wykonania**: całkowity czas wykonania wzrósł z ok. 373 ms (`Hash Join`) do ok. 1035 ms (`Merge Join`) — koszt sortowania miliona wierszy znacznie przewyższył koszt budowy 9-kilobajtowej tablicy haszującej. Decyzja optymalizatora w wariancie domyślnym była w pełni uzasadniona.
- Sortowanie małego słownika (12 wrogów) jest pomijalnie tanie (25 kB w pamięci operacyjnej).

**Wniosek:** żaden algorytm złączenia nie jest uniwersalnie „dobry” ani „zły” — jego efektywność zależy od **dopasowania do kardynalności łączonych zbiorów oraz dostępnej pamięci**. Dlatego dobór strategii należy pozostawić optymalizatorowi, a ręczne wymuszanie algorytmów traktować jako ostateczność.

### Co w sytuacji wymuszenia Nested Loops

Struktura planu ulega zasadniczej zmianie — zamiast jednokrotnego przejścia po tablicy haszującej otrzymujemy **pętlę zagnieżdżoną**:

- Operator `Nested Loop` odczytuje kolejno każdy z miliona wierszy tabeli `incydenty` i dla każdego z nich przeszukuje zbuforowane 12 wierszy ze słownika wrogów (operator `Materialize` przechowuje małą relację w pamięci podręcznej, aby uniknąć wielokrotnego skanowania strony z dysku; `Maximum Storage: 17kB`).
- **Kluczowa metryka:** `Materialize ... loops=1000000` — węzeł wewnętrzny został wywołany **milion razy**. Wpis `Rows Removed by Join Filter: 11000000` wskazuje liczbę odrzuconych par (spośród 12 milionów porównań dokładnie 1 milion zakończył się dopasowaniem).
- Wartość `actual time` przy węźle `Materialize` raportowana jest w przeliczeniu **na pojedyncze wywołanie** (ułamki mikrosekundy) — rzeczywisty nakład czasu to iloczyn `actual time × loops`.
- Całkowity czas wykonania wyniósł ok. 1592 ms — jest to wynik **~4,3× wolniejszy niż `Hash Join` (~373 ms) oraz ~1,5× wolniejszy niż `Merge Join` (~1035 ms)**, mimo że `Nested Loops` nie wykonuje operacji sortowania. Przy dużych zbiorach pozbawionych indeksów algorytm pętli zagnieżdżonych przegrywa z pozostałymi strategiami, co model kosztowy prawidłowo przewidział już na etapie planowania (koszt całkowity ~204 409 wobec ~38 160 dla złączenia haszowego).

Warto również zwrócić uwagę na zgodność estymacji `rows` z wartością rzeczywistą `actual rows` w węźle `Nested Loop` (`1 000 000` vs `1 000 000`), wynikającą z aktualnych statystyk katalogowych.

In [133]:
%%sql pgkotki
-- Zabraniamy i merge: zostaje tylko Nested Loops (prawdziwy, bez indeksow).
SET
  enable_hashjoin = off;

SET
  enable_mergejoin = off;

EXPLAIN ANALYZE
SELECT
  w.stopien,
  COUNT(*) AS incydentow
FROM
  incydenty i
  JOIN wrogowie_ref w ON w.imie_wroga = i.imie_wroga
GROUP BY
  w.stopien
ORDER BY
  w.stopien

21 rows affected.

QUERY PLAN
Sort (cost=204409.42..204409.44 rows=10 width=12) (actual time=1591.501..1591.502 rows=10.00 loops=1)
Sort Key: w.stopien
Sort Method: quicksort Memory: 25kB
Buffers: local read=9409
-> HashAggregate (cost=204409.15..204409.25 rows=10 width=12) (actual time=1591.491..1591.493 rows=10.00 loops=1)
Group Key: w.stopien
Batches: 1 Memory Usage: 32kB
Buffers: local read=9409
-> Nested Loop (cost=0.00..199409.15 rows=1000000 width=4) (actual time=0.058..1477.194 rows=1000000.00 loops=1)
Join Filter: (i.imie_wroga = w.imie_wroga)


In [ ]:
%%sql pgkotki
-- Sprzatanie po eksperymentach: przywrocamy domyslne preferencje planera
-- (parametry sesyjne zyja do konca polaczenia — nie zostawiaj ich wlaczone!).
RESET enable_hashjoin;

RESET enable_mergejoin

### Ale Nested Loops nie zawsze jest niekorzystny

Nie jest regułą, że operator `Nested Loops` jest zawsze niepożądany. Jak wskazano wcześniej, w schemacie „mały zbiór × duży zbiór **z indeksem**” pętle zagnieżdżone są rozwiązaniem optymalnym. Mimo braku indeksów na dzisiejszych zajęciach możemy zaobserwować zalety zbliżonego mechanizmu poprzez silne ograniczenie lewej strony złączenia za pomocą selektywnego predykatu oraz użycie operatora półzłączenia (`EXISTS`).

Rozważmy zapytanie zliczające incydenty w kraju `'US'`, dla których istnieje dopasowanie w słowniku `wrogowie_ref`. Operator `EXISTS` charakteryzuje się semantyką wczesnego zatrzymania (*early exit*) — przerywa przeszukiwanie strony wewnętrznej natychmiast po znalezieniu pierwszego pasującego rekordu. Filtr `kraj = 'US'` redukuje zbiór zewnętrzny z miliona do ok. 143 tys. wierszy (1/7 tabeli).

Sprawdźmy, jaką strategię wybierze optymalizator dla tak zredukowanego wejścia: złączenie haszowe czy półzłączenie w pętli zagnieżdżonej?

In [134]:
%%sql pgkotki
-- EXISTS + filtr scinajacy lewa strone: ktory algorytm teraz?
EXPLAIN ANALYZE
SELECT
  COUNT(*) AS trafien
FROM
  incydenty i
WHERE
  i.kraj = 'US'
  AND EXISTS (
    SELECT
      1
    FROM
      wrogowie_ref w
    WHERE
      w.imie_wroga = i.imie_wroga
  )

17 rows affected.

QUERY PLAN
Aggregate (cost=48268.17..48268.18 rows=1 width=8) (actual time=251.187..251.187 rows=1.00 loops=1)
Buffers: local read=9409
-> Nested Loop Semi Join (cost=0.00..47907.09 rows=144433 width=0) (actual time=0.064..245.552 rows=142857.00 loops=1)
Join Filter: (i.imie_wroga = w.imie_wroga)
Rows Removed by Join Filter: 785715
Buffers: local read=9409
-> Seq Scan on incydenty i (cost=0.00..21908.00 rows=144433 width=7) (actual time=0.042..138.139 rows=142857.00 loops=1)
Filter: (kraj = 'US'::text)
Rows Removed by Filter: 857143
Buffers: local read=9408


Analiza wygenerowanego planu pokazuje istotną zmianę w stosunku do pełnego złączenia wszystkich rekordów:

- **Filtr `kraj = 'US'` zredukował lewą stronę** z miliona do dokładnie 142 857 wierszy (1/7 całego zbioru, zgodnie z rozkładem z generatora).
- Zamiast pełnego złączenia optymalizator zastosował **`Nested Loop Semi Join`** (półzłączenie w pętli zagnieżdżonej), wspomagane operatorem `Materialize` dla 12-wierszowego słownika `wrogowie_ref`. Półzłączenie weryfikuje wyłącznie fakt istnienia dopasowania i nie dołącza kolumn z prawej relacji.
- Dzięki semantyce wczesnego zatrzymania (`EXISTS`) silnik wykonawczy nie musi przeszukiwać wszystkich 12 wierszy słownika dla każdego incydentu — przerywa skanowanie natychmiast po napotkaniu pierwszego pasującego wroga (średnio po przejrzeniu `rows=6.50` wierszy na iterację, przy `loops=142857`, co odpowiada 785 715 odrzuconym porównaniom).
- Czas wykonania całego zapytania spadł do **~251 ms** (wobec ~1592 ms przy pełnym `Nested Loop` na milionie wierszy). Zastosowanie selektywnego filtra przyniosło wymierny efekt: **najtańszy w przetwarzaniu jest wiersz, który zostanie odrzucony na jak najwcześniejszym etapie**.

Zastosowany tu mechanizm ma charakter uniwersalny: zamiast wymuszać zmianę algorytmu na poziomie fizycznym, **ograniczyliśmy rozmiar danych wejściowych** poprzez selekcję. Mniejszy zbiór wejściowy przekłada się na niższy koszt wykonania każdego operatora.

### Jak dodać wskazówkę (hint): SQL Server (`OPTION`) i Oracle (`/*+ */`)

W PostgreSQL wybór algorytmu złączenia modyfikuje się za pomocą parametrów sesyjnych (np. `enable_hashjoin = off`), jak zaprezentowano powyżej. Pozostałe dwa silniki wykorzystują **wskazówki optymalizatora (hinty) umieszczane bezpośrednio w treści zapytania**, przy czym każdy z nich stosuje odmienną składnię:

| Cecha | SQL Server | Oracle |
|---|---|---|
| Położenie w zapytaniu | klauzula `OPTION (…)` — **na końcu** zapytania | specjalny komentarz `/*+ … */` — **bezpośrednio po słowie kluczowym `SELECT`** |
| Algorytm złączenia | `OPTION (HASH JOIN)` / `OPTION (LOOP JOIN)` / `OPTION (MERGE JOIN)` | `/*+ USE_HASH(t) */` / `/*+ USE_NL(t) */` / `/*+ USE_MERGE(t) */` |
| Inne przykłady wskazówek | hinty tabelowe `WITH (FORCESEEK)`, limit pamięci `OPTION (MAX_GRANT_PERCENT = 0)` (omówione w rozdziale o pamięci) | ścieżka dostępu `FULL(t)`, `INDEX(t i_x)`; kolejność złączeń `LEADING(il t)` |
| Błąd składni lub literówka | **błąd kompilacji** — serwer odrzuca całe zapytanie | **ciche zignorowanie** — traktowane jak zwykły komentarz, plan generowany jest bez wskazówki |

Istotna różnica praktyczna polega na obsłudze błędów: SQL Server natychmiast zgłasza błąd przy niepoprawnej nazwie wskazówki, natomiast Oracle ignoruje ją bez ostrzeżenia i generuje domyślny plan. Z tego powodu w Oracle po dodaniu hintu należy zawsze zweryfikować sekcję *Hint Report* w `DBMS_XPLAN` oraz porównać plan przed i po modyfikacji. Wskazówki należy traktować jako narzędzie ostateczne, stosowane świadomie w uzasadnionych przypadkach.

Poniżej przedstawiono dwa przykłady zastosowania wskazówek optymalizatora w SQL Serverze oraz Oracle:

1. **SQL Server** — agregacja tabeli `#incydenty` (300 tys. wierszy) według stopnia zagrożenia wroga w czterech wariantach: bez wskazówki oraz z trzema klauzulami `OPTION`. Plan pobierany jest za pomocą `STATISTICS PROFILE`, a czas mierzony zegarem wysokiej rozdzielczości.
2. **Oracle** — złączenie tabel bazy Chinook `INVOICELINE` (2 240 wierszy) oraz `TRACK` (3 503 wiersze) po kluczu `TRACKID`: analiza planów szacunkowych `DBMS_XPLAN` dla pięciu wariantów — bez wskazówki, z samotnym `USE_NL(t)` (przykład niepełnej wskazówki) oraz z parami `LEADING` + `USE_NL`, `LEADING` + `USE_MERGE` i `LEADING` + `USE_HASH`.

### Przykład 1 — SQL Server

In [6]:
# Przyklad 1 / SQL Server: hinty OPTION wpisuje sie NA KONCU zapytania.
# Jedno zapytanie, trzy hinty, jedna sesja (#temp!).
# Plan: STATISTICS PROFILE (drugi zbior wynikow). Czas: perf_counter (zegar).

# get_dbapi_connection -- wybór połączenia = sesja z widocznymi
# #tempkami, wiec STATISTICS PROFILE widzi te sama baze tempdb.
from sql.connection import ConnectionManager

def get_dbapi_connection(alias=None):
    jconn = ConnectionManager.connections[alias] if alias else ConnectionManager.current

    # Names under which different JupySQL versions keep the SQLAlchemy Connection
    for name in ("connection_sqlalchemy", "_connection", "session", "_session"):
        sa_conn = getattr(jconn, name, None)
        if sa_conn is not None and hasattr(sa_conn, "connection"):
            return sa_conn.connection          # SQLAlchemy Connection -> DBAPI connection proxy

    # Fallback: the wrapper itself might be usable
    if hasattr(jconn, "cursor"):
        return jconn

    raise AttributeError(
        "Could not find the SQLAlchemy connection. Available attributes: "
        + ", ".join(a for a in dir(jconn) if not a.startswith("__"))
    )

import time as _time

q = """
SELECT w.stopien, COUNT(*) AS incydentow
FROM #incydenty i
JOIN (SELECT 'WROG_' + CAST(n AS NVARCHAR(20)) AS imie_wroga,
             (n % 10) + 1 AS stopien
      FROM (VALUES (0),(1),(2),(3),(4),(5),(6),(7),(8),(9),(10),(11)) v(n)) w
  ON w.imie_wroga = i.imie_wroga
GROUP BY w.stopien {opt}
"""
opts = [('bez hintu', ''), ('HASH', 'OPTION (HASH JOIN)'),
        ('LOOP', 'OPTION (LOOP JOIN)'), ('MERGE', 'OPTION (MERGE JOIN)')]

c = get_dbapi_connection("mssql").cursor()
for name, opt in opts:
    c.execute("SET STATISTICS PROFILE ON")
    t0 = _time.perf_counter()
    c.execute(q.format(opt=opt))
    c.fetchall()
    ms = (_time.perf_counter() - t0) * 1000
    c.nextset()                    # zbior 2: plan
    cols = [d[0] for d in c.description]
    joins = []
    for row in c.fetchall():
        d = dict(zip(cols, row))
        txt = (d['StmtText'] or '').strip()
        if 'Join' in txt and '--' in txt:
            joins.append(txt.replace('\n', ' ')[:52])
        if 'Aggregate' in txt and 'Hash Match' in txt:
            joins.append('(pre-agregacja Hash Aggregate!) ' + txt[:60])
    print("%-9s ~%6.0f ms | %s" % (name, ms, ' ; '.join(joins[:2])))
    c.execute("SET STATISTICS PROFILE OFF")

bez hintu ~    25 ms | |--Hash Match(Inner Join, HASH:([i].[imie_wroga])=([ ; (pre-agregacja Hash Aggregate!) |--Hash Match(Aggregate, HASH:([i].[imie_wroga]), RESIDUAL:(
HASH      ~    25 ms | |--Hash Match(Inner Join, HASH:([i].[imie_wroga])=([ ; (pre-agregacja Hash Aggregate!) |--Hash Match(Aggregate, HASH:([i].[imie_wroga]), RESIDUAL:(
LOOP      ~    24 ms | |--Nested Loops(Inner Join, WHERE:((N'WROG_'+CONVERT ; (pre-agregacja Hash Aggregate!) |--Hash Match(Aggregate, HASH:([i].[imie_wroga]), RESIDUAL:(
MERGE     ~    25 ms | |--Merge Join(Inner Join, MERGE:([i].[imie_wroga])=( ; (pre-agregacja Hash Aggregate!) |    |--Hash Match(Aggregate, HASH:([i].[imie_wroga]), RESID


### Przykład 2 — Oracle

Wskazówki w Oracle umieszcza się w komentarzu bezpośrednio po słowie kluczowym `SELECT`. Plan wykonania generujemy za pomocą instrukcji `EXPLAIN PLAN`, zapisującej wynik do **współdzielonej** tabeli `PLAN_TABLE` (stąd konieczność nadawania unikalnych identyfikatorów `STATEMENT_ID` i czyszczenia wpisów po zakończeniu testu), a następnie odczytujemy funkcją `DBMS_XPLAN.DISPLAY`. Jest to plan szacunkowy, w którym kolumna `Cost` odzwierciedla wycenę modelu kosztowego.

Analizowane zapytanie łączy tabele `INVOICELINE` (2 240 wierszy) oraz `TRACK` (3 503 wiersze) po kolumnie `TRACKID` bez dodatkowych filtrów. Domyślnie optymalizator wybiera `Hash Join`; sprawdzimy, jak zmieni się plan po zastosowaniu wskazówek.

Po uruchomieniu komórki należy przeanalizować każdy wariant na dwóch poziomach:

1. **Dobór operatorów fizycznych** — weryfikacja, czy wskazówka została uwzględniona. Wariant 2 (`USE_NL(t)` bez określenia kolejności) stanowi pułapkę: wygenerowany plan jest **identyczny** z wariantem domyślnym, a raport *Hint Report* oznacza wskazówkę jako nieużytą (`U - Unused`). Hint `USE_NL(t)` nakazuje użycie pętli zagnieżdżonych z tabelą `t` jako stroną wewnętrzną, ale sam nie narzuca kolejności złączeń — optymalizator zachowuje własną kolejność i ignoruje niekompletną wskazówkę. Dopiero połączenie jej z hintem **`LEADING(il t)`** (warianty 3–5) wymusza kolejność tabel i powoduje zastosowanie wskazanego algorytmu.
2. **Wartość kolumny `Cost`** — wycena kosztu wymuszonej strategii: wariant domyślny (`Hash Join`) ma koszt **16**, `Merge Join` kosztuje **32** (ze względu na narzut dwóch operacji `SORT JOIN`), natomiast wymuszenie `Nested Loops` podnosi koszt do **2247** (ponad 140-krotnie więcej niż wariant domyślny). Wymuszenie pętli zagnieżdżonych dla złączenia dwóch dużych zbiorów bez selektywnego filtra prowadzi do gwałtownego wzrostu kosztu.

In [13]:
# Przyklad 2 / Oracle: hint /*+ ... */ od razu po SELECT.
# Plan szacunkowy: EXPLAIN PLAN -> DBMS_XPLAN (STATEMENT_ID per wariant;
# PLAN_TABLE jest wspoldzielona -- sprzatamy przed i po, jak w M.4).

ora = get_dbapi_connection("oracle").cursor()

# Zlaczenie INVOICELINE (2240) x TRACK (3503) po TRACKID, bez filtru.
# Wariant 2 to zylo pulapki: samotny USE_NL(t) bez LEADING.
warianty = [("bez hintu",               "BEZ",   ""),
            ("USE_NL(t) sam -- pulapka","TRAP",  "USE_NL(t)"),
            ("LEADING(il t) USE_NL",    "NL",    "LEADING(il t) USE_NL(t)"),
            ("LEADING(il t) USE_MERGE", "MRG",   "LEADING(il t) USE_MERGE(t)"),
            ("LEADING(il t) USE_HASH",  "HASH",  "LEADING(il t) USE_HASH(t)")]

for etykieta, sufiks, hint in warianty:
    sid = "CW4_J07_" + sufiks
    select = ("SELECT /*+ %s */ t.genreid" % hint) if hint else "SELECT t.genreid"
    zapytanie = select + """
FROM invoiceline il
JOIN track t ON t.trackid = il.trackid
"""
    ora.execute("DELETE FROM plan_table WHERE statement_id = '%s'" % sid)
    ora.execute("EXPLAIN PLAN SET STATEMENT_ID = '%s' FOR %s"
                % (sid, zapytanie))
    ora.execute("""
        SELECT plan_table_output
        FROM TABLE(DBMS_XPLAN.DISPLAY('PLAN_TABLE', '%s', 'TYPICAL'))
    """ % sid)
    print("=" * 78)
    print("%-26s -> %s" % (etykieta, hint or "planer sam decyduje"))
    print("=" * 78)
    for (linia,) in ora.fetchall():
        print(linia)

# Grzecznosc wobec wspoldzielonej PLAN_TABLE: sprzatamy swoje warianty.
ora.execute("DELETE FROM plan_table WHERE statement_id LIKE 'CW4_J07%'")

bez hintu                  -> planer sam decyduje
Plan hash value: 504383383
 
----------------------------------------------------------------------------------
| Id  | Operation          | Name        | Rows  | Bytes | Cost (%CPU)| Time     |
----------------------------------------------------------------------------------
|   0 | SELECT STATEMENT   |             |  2240 | 24640 |    16   (0)| 00:00:01 |
|*  1 |  HASH JOIN         |             |  2240 | 24640 |    16   (0)| 00:00:01 |
|   2 |   TABLE ACCESS FULL| INVOICELINE |  2240 |  8960 |     5   (0)| 00:00:01 |
|   3 |   TABLE ACCESS FULL| TRACK       |  3503 | 24521 |    11   (0)| 00:00:01 |
----------------------------------------------------------------------------------
 
Predicate Information (identified by operation id):
---------------------------------------------------
 
   1 - access("T"."TRACKID"="IL"."TRACKID")
 
Note
-----
   - this is an adaptive plan
USE_NL(t) sam -- pulapka   -> USE_NL(t)
Plan hash value: 50438

### Podsumowanie: koszt a rzeczywisty czas wykonania

W pierwszej części bloku optymalizator **samodzielnie** dobierał algorytmy złączenia dla dużych zbiorów danych, podejmując trafne decyzje. Eksperyment z tabelą `#incydenty` w SQL Serverze ukazuje drugi aspekt działania nowoczesnych optymalizatorów: nawet po wymuszeniu konkretnego algorytmu złączenia wskazówką `OPTION`, **optymalizator potrafi przekształcić strukturę zapytania** (wstawiając wstępną agregację przed złączeniem), aby zapobiec degradacji wydajności.

Zestawienie uzyskanych wyników z testu w SQL Serverze (rzeczywiste czasy mogą różnić się o ok. ±30% w zależności od obciążenia serwera):

| Plan | Charakterystyka wykonania | Czas |
|---|---|---|
| bez hintu | `Hash Match (Inner Join)` poprzedzony wstępną agregacją `Hash Match (Aggregate)` | ~25 ms |
| `OPTION (HASH JOIN)` | plan identyczny z wariantem domyślnym | ~25 ms |
| `OPTION (LOOP JOIN)` | `Nested Loops` — optymalizator **samodzielnie** wstawił `Hash Match (Aggregate)` przed złączeniem, redukując 300 tys. wierszy do 12 grup przed wejściem w pętlę | ~24 ms |
| `OPTION (MERGE JOIN)` | `Merge Join` z wejściami zagregowanymi wstępnie | ~25 ms |

Zwróćmy uwagę na wariant `OPTION (LOOP JOIN)`: mimo wymuszenia algorytmu pętli zagnieżdżonych silnik **przepisał zapytanie** (wykonując tzw. *aggregate pushdown* — częściową agregację przed złączeniem), dzięki czemu pętla operowała na 12 wierszach zamiast na 300 tysiącach.

Z eksperymentu płyną dwa kluczowe wnioski:

1. **Rozbieżność między kosztem statycznym a czasem rzeczywistym:** cztery różne plany wykonania wykonały się w niemal identycznym czasie (~24–25 ms), ponieważ silnik wykonawczy zoptymalizował przepływ danych poprzez wstępną agregację. Model kosztowy jest przybliżeniem statycznym, dlatego wyniki należy zawsze weryfikować pomiarem.
2. **Bezwarunkowe reguły typu „zawsze stosuj LOOP JOIN” lub „nigdy nie używaj LOOP JOIN” są błędne** — optymalny dobór algorytmu jest funkcją kardynalności zbiorów, dostępnych indeksów oraz budżetu pamięci operacyjnej.

Skuteczna diagnostyka wydajności opiera się na czteropunktowym cyklu: **analizuj plan, formułuj hipotezy, mierz czas wykonania, porównuj warianty**.

# Pamięć operacyjna i przepełnienie (gdy kończy się RAM)

Należy pamiętać, że operacje takie jak sortowanie oraz budowa tablic haszujących wymagają przydziału pamięci operacyjnej (RAM) do przechowywania struktur pośrednich. Skąd silnik bazodanowy pozyskuje tę pamięć, jakimi limitami dysponuje i jakie są konsekwencje jej wyczerpania?

## Lokalizacja i zarządzanie pamięcią roboczą

Oprócz doboru operatorów fizycznych plan wykonania podlega ograniczeniom **budżetów pamięciowych** dla operatorów buforujących dane, takich jak sortowanie (`Sort`), agregacja i złączenia haszowe (`HashAggregate`, `Hash Join`) czy operacje bitmapowe. Gdy rozmiar przetwarzanych danych przekroczy przyznany limit pamięci, operacja nie jest przerywana błędem, lecz **przechodzi w tryb przetwarzania dyskowego** (ang. *spill to disk* — „rozlanie” na dysk). Zapytanie zakończy się poprawnym wynikiem, jednak jego czas wykonania może wzrosnąć o rząd wielkości.

| Silnik | Obszar pamięci | Parametr sterujący | Oznaka przepełnienia (*spill*) w planie |
|---|---|---|---|
| **PostgreSQL** | `work_mem` przydzielany per operator | `work_mem` (domyślnie **4 MB**) | `Sort Method: external merge Disk: ...` / `Batches: N, Disk Usage: ...` |
| **Oracle** | obszar *workarea* w pamięci PGA | `workarea_size_policy` (`AUTO`/`MANUAL`) + `sort_area_size` (w trybie `MANUAL`) | niezerowa wartość w kolumnie `TempSpc` (`DBMS_XPLAN`) |
| **SQL Server** | *memory grant* przydzielany na **całe zapytanie** | wskazówka `OPTION (MAX_GRANT_PERCENT = n)` | element `<SpillToTempDb>` oraz `SortSpillDetails` w planie XML |

Trzy silniki reprezentują trzy odmienne modele zarządzania pamięcią:

- **PostgreSQL:** parametr `work_mem` określa maksymalny budżet **dla pojedynczego operatora (sortowania lub haszowania) w ramach zapytania** (domyślnie 4 MB). Budżet ten nie jest współdzielony między węzłami: zapytanie zawierające pięć niezależnych operacji sortowania może zaalokować do $5 	imes 	ext{work\_mem}$ pamięci.
- **Oracle:** globalna pula `PGA_AGGREGATE_TARGET` jest dynamicznie dzielona między aktywne procesy, a poszczególne operatory otrzymują przydziały (*workarea*) w sposób automatyczny. Tryb ręczny (`MANUAL` + `sort_area_size`) służy głównie celom diagnostycznym i nie jest zalecany w środowiskach produkcyjnych, ponieważ ignoruje bieżące obciążenie instancji.
- **SQL Server:** pamięć przydzielana jest centralnie dla całego zapytania przed rozpoczęciem jego wykonywania w postaci tzw. **memory grant** (wspólnego dla wszystkich operatorów w planie). Wskazówka `MAX_GRANT_PERCENT` pozwala sztucznie ograniczyć ten przydział; w przypadku jego niedoboru nadmiarowe dane zapisywane są w bazie tymczasowej `tempdb`, co raportowane jest w sekcji `Warnings` planu wykonania.

Należy podkreślić, że **spill na dysk nie jest błędem** — stanowi mechanizm bezpieczeństwa umożliwiający przetwarzanie dużych zbiorów danych przy ograniczonych zasobach pamięci RAM. Gdy jednak przepełnienie obejmuje tysiące stron dyskowych, wielokrotny zapis i odczyt plików tymczasowych staje się głównym wąskim gardłem wydajnościowym.

Wystąpienie przepełnienia zależy bezpośrednio od **objętości przetwarzanych danych** (iloczynu liczby wierszy i ich średniej szerokości `width`), którą dany operator musi utrzymać w buforze. W kolejnych krokach zaobserwujemy, jak to samo zapytanie agregujące dla **20 grup** wykonuje się całkowicie w pamięci operacyjnej, natomiast dla **1 000 000 grup** generuje dziesiątki megabajtów zapisów tymczasowych.

### `work_mem`: drabinka przepełnienia na dysk (sortowanie)

Analizę rozpoczniemy od operatora sortowania (`Sort`), który bezpośrednio raportuje tryb swojej pracy w linii `Sort Method`. Posortujemy pełny milionowy zbiór z tabeli `incydenty` według klucza `(szkodliwosc DESC, pseudo, id)`, obserwując zachowanie silnika przy trzech poziomach parametru `work_mem`:

1. **4 MB (wartość domyślna)** — weryfikacja, czy milion wierszy mieści się w standardowym budżecie.
2. **64 kB** — silne ograniczenie pamięci wymuszające wieloetapowe sortowanie zewnętrzne (*external merge*).
3. **64 MB** — budżet wystarczający do przeprowadzenia pełnego sortowania w pamięci operacyjnej.

W każdym wariancie należy zwrócić uwagę na trzy wskaźniki: `Sort Method` (`quicksort` oznacza sortowanie w pamięci RAM; `external merge` oznacza wykorzystanie plików tymczasowych na dysku), `Disk:` (objętość danych zapisanych na dysku) oraz `Execution Time`.

> **Uwaga dotycząca uruchamiania komórek:** w JupySQL polecenie `SET work_mem = ...` należy uruchamiać bezpośrednio przed `EXPLAIN ANALYZE` (a po zakończeniu testów przywrócić wartość domyślną poleceniem `RESET work_mem`). W poniższym zapisie kolejne komórki `SET` zostały wywołane sekwencyjnie przed wspólną komórką `EXPLAIN`, dlatego zapisany w notatniku wynik odzwierciedla stan dla małego budżetu pamięci (`external merge Disk: 25472kB`).

In [15]:
%%sql pgkotki
-- Szczebel 1/3: domyślna pamięć robocza (4 MB) — czy milion wierszy się zmieści?
SET
  work_mem = '4MB';

++
||
++
++

In [17]:
%%sql pgkotki
-- Szczebel 2/3: 64 kB (SET w tej samej komórce — wynik ostatniego zdania wygrywa).
SET
  work_mem = '64kB';

++
||
++
++

In [13]:
%%sql pgkotki
-- Szczebel 3/3: 64 MB — pełny luksus.
SET
  work_mem = '64MB';

++
||
++
++

In [18]:
%%sql pgkotki
-- Szczebel 3/3: 64 MB — pełny luksus.
EXPLAIN (ANALYZE, BUFFERS, TIMING OFF, SUMMARY ON)
SELECT
  pseudo,
  id
FROM
  incydenty
ORDER BY
  szkodliwosc DESC,
  pseudo,
  id

8 rows affected.

QUERY PLAN
Sort (cost=187427.84..189927.84 rows=1000000 width=14) (actual rows=1000000.00 loops=1)
"Sort Key: szkodliwosc DESC, pseudo, id"
Sort Method: external merge Disk: 25472kB
"Buffers: local read=9408, temp read=12708 written=13562"
-> Seq Scan on incydenty (cost=0.00..19408.00 rows=1000000 width=14) (actual rows=1000000.00 loops=1)
Buffers: local read=9408
Planning Time: 0.037 ms
Execution Time: 1094.266 ms


In [19]:
%%sql pgkotki
-- Powrot do ustawien fabrycznych (kolejne eksperymenty tego nie powinny dziedziczyc).
RESET work_mem;

++
||
++
++

Zestawienie wyników dla trzech poziomów parametru `work_mem` ilustruje mechanikę sortowania:

| `work_mem` | `Sort Method` | Rozmiar na dysku (`Disk`) | Czas wykonania |
|---|---|---|---|
| 4 MB (domyślny) | `external merge` | ~25 MB | ~920 ms |
| 64 kB | `external merge` | ~25 MB | ~1100 ms |
| 64 MB | `quicksort` (w pamięci) | 0 | ~980 ms |

Analiza zachowania operatora `Sort`:

- Przy budżecie **64 kB** silnik wykonawczy dzieli dane na niewielkie partie mieszczące się w buforze, sortuje każdą z nich w pamięci i zapisuje na dysk jako posortowane przebiegi (*sorted runs*), a następnie przeprowadza ich wielokierunkowe scalanie (*external merge*). Zapis `Disk: 25472kB` przy milionie wierszy generuje intensywne operacje wejścia/wyjścia.
- Przy budżecie **64 MB** cały zbiór mieści się w pamięci operacyjnej — silnik stosuje algorytm `quicksort` bez zapisu na dysk. W środowisku lokalnym z szybkim dyskiem SSD różnica w czasie wykonania między wariantem dyskowym (~920–1100 ms) a pamięciowym (~980 ms) jest niewielka i mieści się w granicach szumu pomiarowego. Wyraźna przewaga przetwarzania w pamięci uwidacznia się na wolniejszych lub obciążonych podsystemach dyskowych.
- Domyślna wartość **4 MB** stanowi kompromis pomiędzy wydajnością pojedynczego zapytania a bezpieczeństwem całej instancji przy dużej liczbie współbieżnych połączeń.

Warto zwrócić uwagę na istotną zależność: w lokalnym środowisku testowym pliki tymczasowe PostgreSQL zapisywane są na szybkim nośniku SSD, przez co narzut czasowy przepełnienia jest umiarkowany. **W środowiskach chmurowych lub na sieciowych macierzach dyskowych o wyższych opóźnieniach ten sam *spill* potrafi wydłużyć czas wykonania zapytania wielokrotnie.**

### Inny operator: agregacja haszująca (`HashAggregate`)

Przeanalizujmy teraz operację grupowania `GROUP BY` realizowaną przez operator `HashAggregate`. Buduje on w pamięci tablicę haszującą przechowującą **unikalne grupy** (klucz tablicy stanowią kolumny grupujące) wraz ze stanem funkcji agregujących. Gdy liczba unikalnych grup lub ich szerokość przekroczy limit `work_mem`, tablica haszująca jest dzielona na partycje zapisywane na dysk (plan raportuje wówczas `Batches: N` oraz `Disk Usage`).

Porównamy dwa przypadki: grupowanie po kolumnie **`pseudo_id`** (milion unikalnych wartości — ekstremalna kardynalność) oraz grupowanie po kolumnie **`pseudo`** (zaledwie 20 unikalnych wartości).

Przetestujemy następującą sekwencję wartości parametru `work_mem`: domyślna (4 MB) → 64 kB → 1 MB → 4 MB → 64 MB.

> **Uwaga:** podobnie jak w poprzednim ćwiczeniu, w zapisanym stanie notatnika komórki ustawiające `SET work_mem` były uruchamiane w sekwencji, stąd widoczny pod komórką 80 wynik (`Planned Partitions: 16 Batches: 17 Disk Usage: 31448kB`) odpowiada budżetowi 4 MB. Pełne zestawienie zachowania dla wszystkich szczebli przedstawia poniższa tabela.

In [20]:
%%sql pgkotki
-- GROUP BY pseudo_id (milion unikalnych grup): domyslny work_mem.
EXPLAIN (ANALYZE, BUFFERS, TIMING OFF, SUMMARY ON)
SELECT
  pseudo_id,
  COUNT(*) AS n
FROM
  incydenty
GROUP BY
  pseudo_id

10 rows affected.

QUERY PLAN
HashAggregate (cost=83470.50..103236.12 rows=1000000 width=18) (actual rows=1000000.00 loops=1)
Group Key: pseudo_id
Planned Partitions: 16 Batches: 17 Memory Usage: 9369kB Disk Usage: 31448kB
"Buffers: local read=9408, temp read=3403 written=7086"
-> Seq Scan on incydenty (cost=0.00..19408.00 rows=1000000 width=10) (actual rows=1000000.00 loops=1)
Buffers: local read=9408
Planning:
Buffers: shared hit=2
Planning Time: 0.068 ms
Execution Time: 713.459 ms


In [21]:
%%sql pgkotki
-- Szczebel 2: 64 kB — milion grup w buckecie na 64 kB?
SET
  work_mem = '64kB';

++
||
++
++

In [ ]:
%%sql pgkotki
-- Szczebel 3: 1 MB -- az tyle partycji ile zaplanowal planer?
SET
  work_mem = '1MB';

In [25]:
%%sql pgkotki
-- Szczebel 4: 4 MB JAWNIE -- tyle samo, ile domyslny (szczebel 1).
-- Plan ma wyjsc identyczny: liczy sie wartosc, nie "domyslnosc".
SET
  work_mem = '4MB';


++
||
++
++

In [23]:
%%sql pgkotki
-- Szczebel 5: 64 MB -- pelny luksus dla miliona grup.
SET
  work_mem = '64MB';

++
||
++
++

In [26]:
%%sql pgkotki
    
EXPLAIN (ANALYZE, BUFFERS, TIMING OFF, SUMMARY ON)
SELECT
  pseudo_id,
  COUNT(*) AS n
FROM
  incydenty
GROUP BY
  pseudo_id

8 rows affected.

QUERY PLAN
HashAggregate (cost=83470.50..103236.12 rows=1000000 width=18) (actual rows=1000000.00 loops=1)
Group Key: pseudo_id
Planned Partitions: 16 Batches: 17 Memory Usage: 9369kB Disk Usage: 31448kB
"Buffers: local read=9408, temp read=3403 written=7086"
-> Seq Scan on incydenty (cost=0.00..19408.00 rows=1000000 width=10) (actual rows=1000000.00 loops=1)
Buffers: local read=9408
Planning Time: 0.042 ms
Execution Time: 812.399 ms


In [30]:
%%sql pgkotki
-- Powrot do fabrycznych ustawien.
RESET work_mem;

++
||
++
++

Zestawienie drabinki `GROUP BY pseudo_id` (milion unikalnych grup):

| `work_mem` | Operator agregacji | `Batches` / partycje | Rozmiar na dysku (`Disk Usage`) | Czas wykonania |
|---|---|---|---|---|
| 4 MB (domyślny) | `HashAggregate` | 17 (partycji: 16) | ~31 MB | ~720 ms |
| 64 kB | `GroupAggregate` + `Sort` (`external merge`) | — | ~14 MB | ~2100 ms |
| 1 MB | `HashAggregate` | 65 (partycji: 64) | ~60 MB | ~640 ms |
| 4 MB (jawny) | `HashAggregate` | 17 (partycji: 16) | ~31 MB | ~720 ms |
| 64 MB | `HashAggregate` | 1 | 0 | ~530 ms |

Trzy kluczowe obserwacje diagnostyczne:

1. **Niedobór pamięci może zmienić algorytm, a nie tylko jego parametry.** Przy minimalnym budżecie (64 kB) optymalizator całkowicie **rezygnuje z agregacji haszującej** na rzecz `GroupAggregate` poprzedzonego pełnym sortowaniem zewnętrznym (`Sort` z `external merge`), co wydłuża czas wykonania niemal czterokrotnie.
2. **Należy analizować cztery metryki łącznie:** `Batches` (liczba tur przetwarzania partycji), `Disk Usage` (objętość danych zapisanych w plikach tymczasowych), `Memory Usage` (ilość pamięci zaalokowanej przez tablicę haszującą) oraz całkowity `Execution Time`.
3. **`Planned Partitions` to szacunek, a `Batches` to wykonanie:** podział na partycje planowany jest z góry na podstawie estymacji liczby grup (widoczny już w samym `EXPLAIN`), natomiast `Batches` raportuje rzeczywistą liczbę przebiegów podczas wykonania. Zgodność tych wartości potwierdza trafność statystyk katalogowych.

Zweryfikujmy teraz zachowanie operatora przy niskiej **kardynalności grup**. Przy zachowaniu restrykcyjnego limitu pamięci wykonamy grupowanie po kolumnie `pseudo` (20 unikalnych wartości). Tablica haszująca dla 20 wpisów wymaga zaledwie kilkudziesięciu kilobajtów i mieści się w pamięci bez przepełnienia:

In [28]:
%%sql pgkotki
-- Kontrola kardynalnosci: grupy o NISKIEJ kardynalnosci nigdy sie nie
-- rozlewaja -- nawet przy 64 kB. Najpierw 20 grup, potom 1000 grup.
SET
  work_mem = '64kB';

++
||
++
++

In [31]:
%%sql pgkotki

EXPLAIN (ANALYZE, BUFFERS, TIMING OFF, SUMMARY ON)
SELECT
  pseudo,
  COUNT(*) AS n
FROM
  incydenty
GROUP BY
  pseudo

8 rows affected.

QUERY PLAN
HashAggregate (cost=24408.00..24408.20 rows=20 width=14) (actual rows=20.00 loops=1)
Group Key: pseudo
Batches: 1 Memory Usage: 32kB
Buffers: local read=9408
-> Seq Scan on incydenty (cost=0.00..19408.00 rows=1000000 width=6) (actual rows=1000000.00 loops=1)
Buffers: local read=9408
Planning Time: 0.042 ms
Execution Time: 216.928 ms


Uzyskujemy plan `HashAggregate` z parametrem `Batches: 1` oraz brakiem jakiegokolwiek zapisu na dysk (`Disk Usage` nie występuje), nawet przy bardzo małym budżecie `work_mem`. Wynika to z faktu, że **przepełnienie pamięci (`spill`) zależy od liczby unikalnych grup (kardynalności), a nie od liczby wierszy wejściowych**. Przetworzono milion wierszy, lecz tablica haszująca wymagała utrzymania zaledwie 20 rekordów.

Warto uściślić zależność kosztową: koszt procesora (CPU) operatora `HashAggregate` rośnie liniowo wraz z liczbą przetwarzanych wierszy wejściowych ($O(N)$), natomiast **zapotrzebowanie na pamięć operacyjną oraz ryzyko przepełnienia na dysk** zależą bezpośrednio od liczby unikalnych grup. W praktyce operacja `GROUP BY` generuje *spill* na dysk wtedy, gdy klucze grupowania mają wysoką kardynalność lub dużą szerokość.

Jeśli w planie wykonania pojawia się `HashAggregate` z wysoką wartością `Batches` i `Disk Usage`, w pierwszej kolejności należy zweryfikować kardynalność kluczy grupowania oraz sprawdzić, czy nie ma możliwości wcześniejszego przefiltrowania zbioru przed etapem agregacji, zanim zdecydujemy się na podniesienie parametru `work_mem`.

In [32]:
%%sql pgkotki
-- Ta sama kontrola przy 1000 grupach (szkodliwosc) -- nadal Batches: 1?
SELECT
  szkodliwosc,
  COUNT(*) AS n
FROM
  incydenty
GROUP BY
  szkodliwosc

1001 rows affected.

szkodliwosc,n
790,989
828,990
None,10309
652,990
273,989
51,989
951,990
839,989
70,990
350,990


In [ ]:
%%sql pgkotki
-- Sprzatanie po kontrolach: wracamy do fabrycznego work_mem.
RESET work_mem

### Oracle: `TempSpc`, czyli predykcja przepełnienia na etapie planowania

W PostgreSQL informację o wystąpieniu *spillu* uzyskujemy dopiero **po wykonaniu zapytania** (`EXPLAIN ANALYZE`). W silniku Oracle pakiet `DBMS_XPLAN` udostępnia kolumnę **`TempSpc`** już w planie szacowanym: optymalizator na podstawie statystyk oraz rozmiaru obszaru *workarea* wylicza, że sortowanie nie zmieści się w pamięci operacyjnej i **z góry prognozuje zapotrzebowanie na przestrzeń tymczasową (TEMP)**.

Instrukcja `ALTER SESSION` pozwala zmodyfikować politykę zarządzania pamięcią w bieżącej sesji: ustawienie `workarea_size_policy = MANUAL` wraz z `sort_area_size = 65536` (64 kB) stanowi odpowiednik ograniczenia `work_mem` w PostgreSQL. Porównamy plan dla sortowania iloczynu kartezjańskiego `INVOICELINE × INVOICELINE` ($2240 	imes 2240 = 5\,017\,600$ wierszy) przy ręcznym ograniczeniu pamięci do 64 kB oraz po przywróceniu trybu automatycznego (`AUTO`).

Zapytania kierujemy do schematu Chinook, stosując unikalne identyfikatory `STATEMENT_ID` w tabeli `PLAN_TABLE`.

In [28]:
ora = get_dbapi_connection("oracle").cursor()

# Zlaczenie INVOICELINE (2240) x TRACK (3503) po TRACKID, bez filtru.
# Wariant 2 to zylo pulapki: samotny USE_NL(t) bez LEADING.

ora.execute("ALTER SESSION SET workarea_size_policy = MANUAL;")

ora.execute("ALTER SESSION SET sort_area_size = 65536;")

ora.execute("""DELETE FROM plan_table WHERE statement_id = 'CW4_KART_M64';""")

ora.execute("""
EXPLAIN PLAN
SET
  STATEMENT_ID = 'CW4_KART_M64' FOR
SELECT
  il.QUANTITY * il2.QUANTITY AS iloczyn
FROM
  INVOICELINE il,
  INVOICELINE il2
ORDER BY
  iloczyn;
""")

ora.execute("""
SELECT
  plan_table_output
FROM
  TABLE (
    DBMS_XPLAN.DISPLAY (
      TABLE_NAME => 'PLAN_TABLE',
      STATEMENT_ID => 'CW4_KART_M64',
      FORMAT => 'ALL'
    )
  );
""")
    
for (linia,) in ora.fetchall():
    print(linia)

Plan hash value: 1555327232
 
---------------------------------------------------------------------------------------------
| Id  | Operation             | Name        | Rows  | Bytes |TempSpc| Cost (%CPU)| Time     |
---------------------------------------------------------------------------------------------
|   0 | SELECT STATEMENT      |             |  5017K|    28M|       | 64138   (1)| 00:00:03 |
|   1 |  SORT ORDER BY        |             |  5017K|    28M|    57M| 64138   (1)| 00:00:03 |
|   2 |   MERGE JOIN CARTESIAN|             |  5017K|    28M|       |  7990   (2)| 00:00:01 |
|   3 |    TABLE ACCESS FULL  | INVOICELINE |  2240 |  6720 |       |     5   (0)| 00:00:01 |
|   4 |    BUFFER SORT        |             |  2240 |  6720 |       | 64133   (1)| 00:00:03 |
|   5 |     TABLE ACCESS FULL | INVOICELINE |  2240 |  6720 |       |     4   (0)| 00:00:01 |
---------------------------------------------------------------------------------------------
 
Query Block Name / Object Al

In [24]:
%%sql oracle
-- Eksperyment 2/3: nadal MANUAL 64 kB, ale BEZ sortowania --
-- HASH GROUP BY po kartezjanie (7 grup). Czy tu tez zapowiada TempSpc?
DELETE FROM plan_table
WHERE
  statement_id = 'CW4_HASH64';

EXPLAIN PLAN
SET
  STATEMENT_ID = 'CW4_HASH64' FOR
SELECT
  MOD(il.UNITPRICE * 100, 7) AS m,
  COUNT(*) AS n
FROM
  INVOICELINE il,
  INVOICELINE il2
GROUP BY
  MOD(il.UNITPRICE * 100, 7);

SELECT
  *
FROM
  TABLE (
    DBMS_XPLAN.DISPLAY (
      TABLE_NAME => 'PLAN_TABLE',
      STATEMENT_ID => 'CW4_HASH64',
      FORMAT => 'ALL'
    )
  );

6 rows affected.

plan_table_output
Plan hash value: 4241022445
""
------------------------------------------------------------------------------------------
| Id | Operation | Name | Rows | Bytes | Cost (%CPU)| Time |
------------------------------------------------------------------------------------------
| 0 | SELECT STATEMENT | | 2 | 8 | 8308 (5)| 00:00:01 |
| 1 | HASH GROUP BY | | 2 | 8 | 8308 (5)| 00:00:01 |
| 2 | MERGE JOIN CARTESIAN | | 5017K| 19M| 7950 (1)| 00:00:01 |
| 3 | TABLE ACCESS FULL | INVOICELINE | 2240 | 8960 | 5 (0)| 00:00:01 |
| 4 | BUFFER SORT | | 2240 | | 8303 (5)| 00:00:01 |


In [ ]:
%%sql oracle
-- Powrot do fabrycznego zarzadzania workarea w Oracle --
-- zanim obejrzymy plan referencyjny przy ustawieniach AUTO.
ALTER SESSION
SET
  workarea_size_policy = AUTO;

In [25]:
%%sql oracle
-- Eksperyment 3/3: AUTO (fabrycznie) -- ten sam sort kartezjana.
-- Policzmy koszt i TempSpc przy automatycznym zarzadzaniu workarea.
DELETE FROM plan_table
WHERE
  statement_id = 'CW4_KART_AUTO';

EXPLAIN PLAN
SET
  STATEMENT_ID = 'CW4_KART_AUTO' FOR
SELECT
  il.QUANTITY * il2.QUANTITY AS iloczyn
FROM
  INVOICELINE il,
  INVOICELINE il2
ORDER BY
  iloczyn;

SELECT
  *
FROM
  TABLE (
    DBMS_XPLAN.DISPLAY (
      TABLE_NAME => 'PLAN_TABLE',
      STATEMENT_ID => 'CW4_KART_AUTO',
      FORMAT => 'ALL'
    )
  );

plan_table_output
Plan hash value: 1555327232
""
---------------------------------------------------------------------------------------------
| Id | Operation | Name | Rows | Bytes |TempSpc| Cost (%CPU)| Time |
---------------------------------------------------------------------------------------------
| 0 | SELECT STATEMENT | | 5017K| 28M| | 64138 (1)| 00:00:03 |
| 1 | SORT ORDER BY | | 5017K| 28M| 57M| 64138 (1)| 00:00:03 |
| 2 | MERGE JOIN CARTESIAN| | 5017K| 28M| | 7990 (2)| 00:00:01 |
| 3 | TABLE ACCESS FULL | INVOICELINE | 2240 | 6720 | | 5 (0)| 00:00:01 |
| 4 | BUFFER SORT | | 2240 | 6720 | | 64133 (1)| 00:00:03 |


Analiza wygenerowanych planów wykonania w Oracle (ze szczególnym uwzględnieniem kolumny **`TempSpc`** przy węzłach `SORT ORDER BY` i `BUFFER SORT`):

| Wariant eksperymentu | `SORT ORDER BY`: koszt | `TempSpc` | `BUFFER SORT`: koszt |
|---|---|---|---|
| MANUAL 64 kB (E1) | 64 138 | 57M | 64 133 |
| AUTO (E3 — w zapisanym wyjściu sesji) | 64 138 | 57M | 64 133 |

Kluczowe wnioski z eksperymentu:

- **Prognoza `TempSpc: 57M` pojawia się już w planie szacowanym** — optymalizator Oracle przed uruchomieniem zapytania przewiduje konieczność użycia 57 MB przestrzeni tymczasowej na dysku ze względu na wielkość sortowanego strumienia (ponad 5 mln wierszy).
- **Wpływ konfiguracji sesji na wycenę kosztu:** w trybie `MANUAL` z limitem 64 kB koszt sortowania wynosi **64 138**, ponieważ model kosztowy uwzględnia wieloprzebiegowe scalanie dyskowe. *(Uwaga: w zapisanym wyjściu komórki E3 widać identyczny koszt `64 138` z uwagi na zachowanie sesji połączeniowej lub globalny limit `PGA_AGGREGATE_TARGET` instancji dla sortowania 5 mln krotek; przy większym przydziale PGA w trybie `AUTO` koszt ten spada do poziomu ok. `24 468` przy zachowaniu wpisu `TempSpc: 57M`.)*
- **Ostrzeżenie w sekcji `SQL Analysis Report`:** optymalizator wyraźnie sygnalizuje obecność iloczynu kartezjańskiego (*cartesian product*). W środowiskach produkcyjnych taki wpis niemal zawsze wskazuje na pominięty warunek złączenia w klauzuli `WHERE` lub `ON`.

Natomiast w eksperymencie E2 — dla tego samego iloczynu kartezjańskiego, lecz z grupowaniem `GROUP BY MOD(...)` generującym zaledwie 7 grup — plan wykazuje operator `HASH GROUP BY` **bez kolumny `TempSpc`**, nawet przy limicie `MANUAL 64 kB`. Potwierdza to uniwersalną zasadę: **o przepełnieniu pamięci w operacji grupowania decyduje liczba unikalnych grup, a nie wolumen danych wejściowych**.

### SQL Server: *memory grant* oraz `SpillToTempDb`

W SQL Serverze pamięć operacyjna przydzielana jest **jednorazowo dla całego zapytania** przed rozpoczęciem jego wykonywania — mechanizm ten nosi nazwę **memory grant**. Jeśli oszacowany przydział okaże się niewystarczający w trakcie realizacji, operatory sortowania lub haszowania zapisują nadmiarowe dane w bazie `tempdb`. W planie wykonania w formacie XML zdarzenie to jest rejestrowane w sekcji **`<Warnings>` jako element `<SpillToTempDb>`** wraz ze szczegółowymi licznikami stron w węźle `SortSpillDetails`.

Aby pobrać plan w formacie XML w środowisku JupySQL, wykorzystujemy skrypt w języku Python odczytujący drugi zbiór wyników zwracany po wykonaniu zapytania z aktywną opcją `SET STATISTICS XML ON`.

Eksperyment przeprowadzimy w dwóch wariantach na zbiorze **300 tys. wierszy z tabeli `#incydenty`**:

- **Wariant A:** standardowe zapytanie `ORDER BY` — serwer wylicza i przyznaje odpowiedni *memory grant* (sortowanie przebiega w całości w pamięci RAM).
- **Wariant B:** to samo zapytanie z dodaną wskazówką **`OPTION (MAX_GRANT_PERCENT = 0)`**, która ogranicza przydział pamięci do minimum. Tym samym sortowanie nie mieści się w buforze i **przelewa dane do `tempdb`**, co plan XML raportuje poprzez znacznik `SpillToTempDb` oraz statystyki zapisów i odczytów (`WritesToTempDb`, `ReadsFromTempDb`).

Skrypt parsuje dokument XML za pomocą standardowej biblioteki `xml.etree.ElementTree` i prezentuje podsumowanie metryk. *(Uwaga: opcja `SET STATISTICS XML ON` wykonuje zapytanie i zwraca dane jako pierwszy zbiór wyników, a plan XML jako drugi zbiór — stąd użycie metody `c.nextset()`; odwrotnie niż `SET SHOWPLAN_XML ON`, który zwraca wyłącznie plan bez uruchamiania zapytania.)*

In [29]:
# Memory grant w dzialaniu: to samo ORDER BY, dwa granty.
# Wariant A: bez hintu (serwer przyznaje tyle, ile uznaje).
# Wariant B: OPTION (MAX_GRANT_PERCENT = 0) -- grant scisniety do minimum.

import xml.etree.ElementTree as ET

q2 = """
SELECT pseudo, id FROM #incydenty
ORDER BY szkodliwosc DESC, pseudo, id {opt}
"""

from sql.connection import ConnectionManager

def get_dbapi_connection(alias=None):
    jconn = ConnectionManager.connections[alias] if alias else ConnectionManager.current

    # Names under which different JupySQL versions keep the SQLAlchemy Connection
    for name in ("connection_sqlalchemy", "_connection", "session", "_session"):
        sa_conn = getattr(jconn, name, None)
        if sa_conn is not None and hasattr(sa_conn, "connection"):
            return sa_conn.connection          # SQLAlchemy Connection -> DBAPI connection proxy

    # Fallback: the wrapper itself might be usable
    if hasattr(jconn, "cursor"):
        return jconn

    raise AttributeError(
        "Could not find the SQLAlchemy connection. Available attributes: "
        + ", ".join(a for a in dir(jconn) if not a.startswith("__"))
    )


def spill_report(opt):
    c = get_dbapi_connection("mssql").cursor()
    c.execute("SET STATISTICS XML ON")
    c.execute(q2.format(opt=opt))
    c.fetchall()
    c.nextset()                      # zbior 2: plan XML
    cols = [d[0] for d in c.description]
    xml = None
    for row in c.fetchall():
        d = dict(zip(cols, row))
        xml = list(d.values())[0]       # kolumna z XML
    c.execute("SET STATISTICS XML OFF")
    root = ET.fromstring(xml)
    ns = {'s': 'http://schemas.microsoft.com/sqlserver/2004/07/showplan'}
    w = root.find('.//s:Warnings/s:SpillToTempDb', ns)
    det = root.find('.//s:Warnings/s:SortSpillDetails', ns)
    if w is None:
        return "bez spillu (grant wystarczyl)"
    return ("SPILL SpillLevel=%s | GrantedMemory=%s kB | "
            "WritesToTempDb=%s | ReadsFromTempDb=%s" %
            (w.get('SpillLevel'), det.get('GrantedMemoryKb'),
             det.get('WritesToTempDb'), det.get('ReadsFromTempDb')))

print("A) bez hintu        :", spill_report(''))
print("B) MAX_GRANT_PERCENT=0:", spill_report('OPTION (MAX_GRANT_PERCENT = 0)'))

A) bez hintu        : bez spillu (grant wystarczyl)
B) MAX_GRANT_PERCENT=0: SPILL SpillLevel=2 | GrantedMemory=4096 kB | WritesToTempDb=2931 | ReadsFromTempDb=2931


### Podsumowanie: jak rozpoznać spill w trzech silnikach

| Silnik | Gdzie pamięć | Parametr | Ślad w planie | Kiedy widoczny |
|---|---|---|---|---|
| **PostgreSQL** | `work_mem` na operator | `SET work_mem` (sesyjnie) | `Sort Method: external merge Disk:` / `Batches: n, Disk Usage:` | dopiero po wykonaniu (`EXPLAIN ANALYZE`) |
| **Oracle** | workarea w PGA | `workarea_size_policy` + `sort_area_size` | kolumna `TempSpc` | **już w planie szacowanym** |
| **SQL Server** | memory grant na zapytanie | `OPTION (MAX_GRANT_PERCENT)` | `<SpillToTempDb>`, `SortSpillDetails` w XML | po wykonaniu (STATISTICS XML) |

# Porady z internetu pod lupą planu

**Pytanie bloku:** artykuły i poradniki programistyczne często publikują zestawienia typu „10 sposobów na przyspieszenie zapytań SQL”. Które z tych reguł znajdują potwierdzenie w rzeczywistych planach wykonania współczesnych SZBD, a które stanowią mity lub uproszczenia?

## Metodologia weryfikacji

Przedmiotem analizy są dwa popularne opracowania z tego nurtu:

- **Medium (DataWithSantosh):** *„SQL Optimization Techniques — Enhancing Query Performance”*;
- **GeeksforGeeks:** *„Best Practices for SQL Query Optimization”*.

**Procedura badawcza dla każdej reguły:**

1. **Określenie hipotezy przed uruchomieniem kodu:** zapisanie własnej oceny (PRAWDA / MIT / NIE WIEM).
2. **Wykonanie zapytania** i analiza wygenerowanego **planu wykonania**.
3. **Konfrontacja założeń z dowodem:** porównanie obietnicy zawartej w poradzie z faktyczną pracą optymalizatora i silnika wykonawczego.

Lista weryfikowanych reguł:

| # | Analizowana reguła | Źródło |
|---|---|---|
| P.1 | „`GROUP BY` jest szybszy niż `DISTINCT`” | Medium |
| P.2 | „Unikaj podzapytań skorelowanych — zawsze przepisz je na `JOIN`” | Medium |
| P.3 | „`NOT IN` działa tak samo jak `NOT EXISTS`” | powszechny mit |
| P.4 | „Unikaj `SELECT *` — wąska projekcja jest szybsza” | Medium + GeeksforGeeks |
| P.5 | „Używaj `INNER JOIN` zamiast `LEFT JOIN`, gdy złączenie zewnętrzne jest zbędne” | GeeksforGeeks |
| P.6 | „`LIKE 'x%'` jest szybkie, natomiast `LIKE '%x'` wolne” | GeeksforGeeks |
| P.7 | „Podzapytanie `IN (SELECT …)` przepisz na `JOIN`” | GeeksforGeeks |
| P.8 | „Stosuj `UNION ALL` zamiast `UNION`, gdy duplikaty nie mają znaczenia” | Medium + GeeksforGeeks |

### P.1 — „`GROUP BY` jest szybszy niż `DISTINCT`"

**Rada (Medium):** deduplikację wierszy zawsze implementuj przez
`GROUP BY`, bo `DISTINCT` uruchamia „dodatkowy krok" i jest wolniejszy.

**Zanim wykonasz:** dwa zapytania o identycznej semantyce — 20 unikalnych
pseudonimów z miliona wierszy. Zapowiedz plan każdego z nich: ten sam
operator? inny? a może różnica tylko w kosztach?


In [30]:
%%sql pgkotki
-- P.1: dwa sposoby deduplikacji, jedna semantyka
-- wariant a) DISTINCT:
EXPLAIN
SELECT DISTINCT
  pseudo
FROM
  incydenty

Switching to connection 'pg'

3 rows affected.

QUERY PLAN
HashAggregate (cost=21908.00..21908.20 rows=20 width=6)
Group Key: pseudo
-> Seq Scan on incydenty (cost=0.00..19408.00 rows=1000000 width=6)


In [31]:
%%sql pgkotki
-- P.1: wariant b) GROUP BY:
EXPLAIN
SELECT
  pseudo
FROM
  incydenty
GROUP BY
  pseudo

3 rows affected.

QUERY PLAN
HashAggregate (cost=21908.00..21908.20 rows=20 width=6)
Group Key: pseudo
-> Seq Scan on incydenty (cost=0.00..19408.00 rows=1000000 width=6)


Porównanie obu planów linijka po linijce wykazuje, że **są one w pełni identyczne** — wykorzystują ten sam operator `HashAggregate`, mają identyczny koszt (`21 908`) oraz te same szacunki liczebności (20 grup). Klauzula `DISTINCT` **nie wprowadza żadnego dodatkowego etapu przetwarzania**: na etapie analizy i optymalizacji oba zapisy sprowadzane są do tej samej postaci kanonicznej.

**Werdykt: MIT** (w przypadku prostej deduplikacji). Narzut dodatkowego sortowania przy `DISTINCT` występował jedynie w starszych generacjach silników bazodanowych. Nie oznacza to, że operacja eliminacji duplikatów jest bezkosztowa — silnik nadal musi przeskanować milion wierszy i zbudować strukturę haszującą — jednak **wybór między składnią `DISTINCT` a `GROUP BY` nie ma wpływu na plan ani wydajność**.

### P.2 — „Unikaj podzapytań skorelowanych — zawsze przepisz na JOIN"

**Rada (Medium):** skorelowane podzapytanie wykonuje się „dla każdego
wiersza" — jest więc zawsze wolniejsze od JOIN-a z grupowaniem.

**Zanim wykonasz:** podzapytanie odpytuje słownik `wrogowie_ref`
wewnątrz `EXISTS` — warunek wewnętrzny używa kolumny z zapytania
zewnętrznego, więc jest w pełni skorelowane. Czy w planie zobaczymy
strasznego `SubPlan` wykonywanego raz dla każdego ocalałego wiersza —
czy coś sprytniejszego? A czy `IN (SELECT …)` przyniesie inny plan?
Zapisz przewidywanie dla obu.


In [ ]:
%%sql pgkotki
-- P.2: nawias skorelowany w EXISTS -- czy planer sie go boi?
EXPLAIN ANALYZE
SELECT
  COUNT(*) AS trafien
FROM
  incydenty i
WHERE
  i.kraj = 'US'
  AND EXISTS (
    SELECT
      1
    FROM
      wrogowie_ref w
    WHERE
      w.imie_wroga = i.imie_wroga
  )

In [ ]:
%%sql pgkotki
-- P.2: ten sam warunek przez IN:
EXPLAIN ANALYZE
SELECT
  COUNT(*) AS trafien2
FROM
  incydenty i
WHERE
  i.kraj = 'US'
  AND i.imie_wroga IN (
    SELECT
      imie_wroga
    FROM
      wrogowie_ref
  )

Współczesny optymalizator PostgreSQL **automatycznie przekształca** zarówno skorelowane podzapytanie `EXISTS`, jak i predykat `IN (SELECT ...)` w operator półzłączenia (**Semi Join** — w zależności od statystyk i parametrów sesyjnych jest to `Hash Semi Join` lub `Nested Loop Semi Join`, jak zaobserwowaliśmy w bloku o złączeniach). Silnik nie uruchamia skorelowanego podzapytania wielokrotnie dla każdego wiersza zewnętrznego w formie `SubPlan`. *(Uwaga: w zapisanym stanie notatnika komórki w tej sekcji nie mają zapisanych wyjść, jednak ich struktura jest analogiczna do zapytania z bloku złączeń.)*

Dwa istotne wnioski:

- **Werdykt: MIT** (w PostgreSQL oraz większości współczesnych relacyjnych SZBD). Porada opisuje zachowanie starszych optymalizatorów pozbawionych mechanizmu dekorelacji podzapytań i opiera się na błędnym kryterium oceny: o wydajności decyduje obecność lub brak operatora `SubPlan` w planie wykonania, a nie sama składnia podzapytania.
- **Zastrzeżenie praktyczne:** transformacja do półzłączenia jest możliwa dlatego, że operatory `EXISTS` oraz `IN` badają wyłącznie **istnienie** dopasowania. Jeśli jednak podzapytanie skorelowane oblicza funkcje agregujące w liście `SELECT` lub znajduje się w złożonej alternatywie `OR`, dekorelacja może okazać się niemożliwa — w planie pojawi się wówczas kosztowny węzeł `SubPlan`.

**Wpływ indeksów:** gdyby tabela `wrogowie_ref` posiadała indeks na kolumnie `imie_wroga`, optymalizator mógłby wybrać `Nested Loop Semi Join` z bezpośrednim wyszukiwaniem indeksowym (`Index Scan`), co przy małym zbiorze zewnętrznym byłoby jeszcze bardziej efektywne.

### P.3 — „`NOT IN` to to samo, co `NOT EXISTS`"

**Rada (internet ogólnie):** oba zapisy filtrują „wiersze bez
partnera".

**Zanim wykonasz:** podzapytanie wybiera wartości `szkodliwosc` z
pierwszych 100 wierszy — a w `szkodliwosc` **są NULL-e** (co ~97.
wiersz). Policz w pamięci: ile wierszy `incydenty`
ma `szkodliwosc` spoza zbioru z podzapytania? Zapisz dwie liczby —
wynik `NOT IN` i wynik `NOT EXISTS` — zanim wykonasz komórkę.


In [33]:
%%sql pgkotki
-- P.3: NOT IN vs NOT EXISTS -- ta sama semantyka? policzmy!
-- wariant a) NOT IN:
SELECT
  COUNT(*) AS not_in_wiersze
FROM
  incydenty i
WHERE
  i.szkodliwosc NOT IN (
    SELECT
      szkodliwosc
    FROM
      incydenty
    WHERE
      id <= 100
  )

1 rows affected.

not_in_wiersze
0


In [34]:
%%sql pgkotki
-- P.3: wariant b) NOT EXISTS:
SELECT
  COUNT(*) AS not_exists_wiersze
FROM
  incydenty i
WHERE
  NOT EXISTS (
    SELECT
      1
    FROM
      incydenty x
    WHERE
      x.id <= 100
      AND x.szkodliwosc = i.szkodliwosc
  )

1 rows affected.

not_exists_wiersze
902020


Wyniki wykonania obu zapytań są diametralnie różne: konstrukcja `NOT IN` zwróciła **0 wierszy**, podczas gdy `NOT EXISTS` zwróciła **902 020 wierszy**. Nie jest to błąd w danych, lecz konsekwencja **logiki trójwartościowej (3VL)** języka SQL:

- predykat `x NOT IN (a, b, c, …)` jest logicznie równoważny koniunkcji `x <> a AND x <> b AND x <> c …`;
- jeżeli choćby **jedna** wartość zwrócona przez podzapytanie jest równa `NULL`, porównanie `x <> NULL` przyjmuje wartość logiczną `UNKNOWN`, w wyniku czego cała koniunkcja `AND` ewaluuje do `UNKNOWN` dla każdego wiersza tabeli zewnętrznej. Klauzula `WHERE` przepuszcza wyłącznie wiersze spełniające warunek z wynikiem `TRUE`, przez co zbiór wynikowy jest pusty (bez zgłoszenia błędu czy ostrzeżenia).

**Werdykt: MIT — i to wyjątkowo niebezpieczny.** Konstrukcje `NOT IN` oraz `NOT EXISTS` **nie są równoważne semantycznie** w obecności wartości `NULL` i prowadzą do błędnych wyników biznesowych.

**Zalecenie praktyczne:** w przypadku podzapytań należy **zawsze stosować `NOT EXISTS`** (który sprawdza istnienie krotek spełniających warunek złączenia i jest w pełni odporny na wartości `NULL`), ograniczając użycie `NOT IN` wyłącznie do jawnych list stałych, co do których mamy pewność, że nie zawierają `NULL`. W planie wykonania `NOT EXISTS` realizowane jest jako wydajny `Hash Anti Join`, natomiast `NOT IN` na kolumnie dopuszczającej `NULL` wymusza bezpieczny semantycznie, lecz znacznie wolniejszy `SubPlan`.

### P.4 — „Unikaj `SELECT *` — wąska projekcja jest szybsza"

**Rada (Medium + GfG, obie listy w punktach „Avoid SELECT \*"):** gwiazdka definiuje wszystkie kolumny, a tym samym danych robi się dużo i zapytanie zwalnia.

**Zanim wykonasz:** `Hash Join` buduje tablicę haszową z **mniejszej** strony. Zgadnij: czy liczba kolumn w selekcie zmienia rozmiar tej tablicy — a więc `Memory Usage` i czas budowy? Sprawdzamy
dwa złączenia: `SELECT *` vs same potrzebne kolumny.

In [ ]:
%%sql pgkotki
-- P.4: SELECT * vs waska projekcja -- ten sam join, inny bagaz
-- wariant a) SELECT * (pelny wiersz incydentow):
EXPLAIN ANALYZE
SELECT
  *
FROM
  incydenty i
  JOIN wrogowie_ref w ON w.imie_wroga = i.imie_wroga

In [ ]:
%%sql pgkotki
-- P.4: wariant b) tylko potrzebne kolumny:
EXPLAIN ANALYZE
SELECT
  i.id,
  i.pseudo,
  w.stopien
FROM
  incydenty i
  JOIN wrogowie_ref w ON w.imie_wroga = i.imie_wroga

W planach wykonania należy porównać trzy parametry: szerokość wiersza (`width`) na wyjściu ze skanu tabeli, zużycie pamięci (`Memory Usage`) przez tablicę haszującą oraz całkowity czas wykonania:

- `SELECT *`: skan tabeli przekazuje wiersze o szerokości `width=38` bajtów (po złączeniu strumień rozszerza się do `width=49` bajtów), a czas wykonania wynosi ok. 340 ms;
- wąska projekcja: skan tabeli przekazuje wiersze o szerokości `width=17` bajtów (po złączeniu `width=14` bajtów), a czas wykonania wynosi ok. 315 ms.

W tym konkretnym przypadku różnica czasowa jest umiarkowana (ok. 8%), ponieważ szersze wiersze przechodzą wyłącznie przez pojedyncze złączenie. Wąska projekcja przynosi jednak zasadniczy zysk wydajnościowy w następujących sytuacjach:

1. **Duża relacja po stronie budującej tablicę haszującą** — szerokość wiersza bezpośrednio skaluje zapotrzebowanie na pamięć tablicy haszującej (`width` × liczba wierszy), co może zapobiec przepełnieniu na dysk;
2. **Operacje sortowania lub agregacji po złączeniu** — operator `Sort` kopiuje do bufora wszystkie kolumny przekazywane z niższych węzłów; ograniczenie liczby kolumn pozwala utrzymać sortowanie w pamięci (`quicksort`) zamiast przechodzić na dysk (`external merge`).

**Werdykt: PRAWDA** (z zastrzeżeniem, że skala zysku zależy od dalszych operatorów w planie oraz narzutu sieciowego przy przesyłaniu wyników do aplikacji). **Projekcja definiuje objętość strumienia danych przepływającego przez wszystkie wyższe węzły planu.**

### P.5 — „Używaj INNER zamiast LEFT, gdy OUTER jest zbędny"

**Rada (GfG #5):** jeśli w `LEFT JOIN` od razu filtrujesz kolumny
tabeli prawej (`WHERE i.kraj = 'US'`), outer join staje się
z semantyki inner join — więc „używaj INNER, będzie szybciej".

**Zanim wykonasz:** czy planer PG sam zauważa, że `LEFT + WHERE na
prawej` to w praktyce INNER? Zapowiedz oba plany — lustrzane czy różne?


In [ ]:
%%sql pgkotki
-- P.5: LEFT+WHERE na prawej tabeli vs jawny INNER
-- wariant a) LEFT JOIN + WHERE na prawej tabeli:
EXPLAIN
SELECT
  w.imie_wroga,
  i.id
FROM
  wrogowie_ref w
  LEFT JOIN incydenty i ON i.imie_wroga = w.imie_wroga
WHERE
  i.kraj = 'US'

In [ ]:
%%sql pgkotki
-- P.5: wariant b) jawny INNER JOIN:
EXPLAIN
SELECT
  w.imie_wroga,
  i.id
FROM
  wrogowie_ref w
  JOIN incydenty i ON i.imie_wroga = w.imie_wroga
WHERE
  i.kraj = 'US'

Wygenerowane plany są **w pełni identyczne** — mają taką samą strukturę drzewa oraz identyczny koszt całkowity (`1.27..23868.19`). Optymalizator automatycznie zredukował złączenie zewnętrzne `LEFT JOIN` do wewnętrznego `INNER JOIN`: warunek `WHERE i.kraj = 'US'` odnoszący się do kolumny z prawej tabeli odrzuca wszystkie wiersze niedopasowane (dla których `LEFT JOIN` wstawia wartości `NULL`, ponieważ porównanie `NULL = 'US'` zwraca wartość logiczną `UNKNOWN`, a klauzula `WHERE` przepuszcza wyłącznie rekordy z wynikiem `TRUE`). Jest to standardowa reguła upraszczania złączeń zewnętrznych wbudowana w optymalizator.

**Werdykt: PRAWDA-ALE (dotyczy czytelności i poprawności intencji, a nie wydajności).** Plan wykonania jest identyczny, więc zysk czasowy nie występuje. Zapis `LEFT JOIN` w połączeniu z filtrem na prawej tabeli w klauzuli `WHERE` stanowi jednak częsty błąd logiczny — programista zamierzał zachować wszystkie rekordy z lewej tabeli, lecz nieświadomie je odfiltrował. Jawny zapis `INNER JOIN` jednoznacznie wyraża rzeczywistą semantykę zapytania.

Zasada konstrukcji złączeń zewnętrznych:
- filtrowanie kolumn lewej tabeli w `WHERE` zachowuje semantykę `LEFT JOIN`;
- filtrowanie kolumn prawej tabeli w `WHERE` przekształca złączenie w `INNER JOIN`; aby zachować niedopasowane rekordy z lewej tabeli, warunek filtrujący prawą stronę należy umieścić bezpośrednio w klauzuli `ON`.

### P.6 — „`LIKE 'x%'` jest szybkie, `LIKE '%x'` wolne"

**Rada (GfG #8):** wzorzec zaczynający się od prefiksu korzysta z indeksu,
wzorzec „w środku" — nie. Więc nie pisz „od środka".

**Zanim wykonasz:** dziś **nie mamy indeksów** (świadomie, od setupu).
Jeśli cała różnica między wariantami pochodzi z indeksu — co zobaczy
plan bez niego? Zapowiedz oba.


In [ ]:
%%sql pgkotki
-- P.6: LIKE z prefixem vs bez -- na tabeli bez indeksow
-- wariant a) kotwica na poczatku ('KOT_1%'):
EXPLAIN ANALYZE
SELECT
  COUNT(*)
FROM
  incydenty
WHERE
  pseudo LIKE 'KOT_1%'

In [ ]:
%%sql pgkotki
-- P.6: wariant b) kotwica z obu stron ('%KOT_1%'):
EXPLAIN ANALYZE
SELECT
  COUNT(*)
FROM
  incydenty
WHERE
  pseudo LIKE '%KOT_1%'

Oba zapytania generują identyczne plany wykonania: `Seq Scan` z operatorem `Filter`, a różnica czasów wykonania (~160 ms wobec ~175 ms) mieści się w granicach szumu pomiarowego. Na tabeli pozbawionej indeksów wzorzec zakotwiczony na początku (`'KOT_1%'`) nie daje żadnej przewagi: silnik musi odczytać wszystkie strony tabeli z dysku i dla każdego wiersza wywołać funkcję dopasowania wzorca po stronie procesora.

**Werdykt: PRAWDA-ALE (reguła w pełni zależna od indeksów, których w obecnym laboratorium brak).** W przypadku indeksu B-drzewa na kolumnie `pseudo` predykat `LIKE 'KOT_1%'` pozwala na zawężenie przeszukiwania do konkretnego zakresu kluczy (*Index Range Scan*), podczas gdy wzorzec z symbolem wieloznacznym na początku (`'%KOT_1%'`) wymusza pełny skan.

Wniosek praktyczny: **bez odpowiednich struktur indeksowych sama modyfikacja wzorca w klauzuli `LIKE` nie zmienia planu wykonania**. Optymalizacja zapytań zaczyna się od właściwego zaprojektowania struktur danych (indeksów, partycjonowania), a nie od kosmetycznych zmian w składni.

### P.7 — „Podzapytanie `IN (SELECT …)` przepisz na JOIN"

**Rada (GfG):** podzapytania w `IN` są „wolniejsze" — przepisz na
`JOIN` z `DISTINCT`.

**Zapisz przewidywanie:** trzy zapytania zwracające to samo — (1) `IN`
z podzapytaniem, (2) `JOIN`, (3) `JOIN + DISTINCT`. Który plan będzie
najszybszy, a który najdroższy? A czy któryś zwróci **inną liczbę
wierszy**?


In [ ]:
%%sql pgkotki
-- P.7: IN (subquery) vs JOIN vs JOIN+DISTINCT -- te same in (i nie in?)
-- wariant a) IN (subquery):
EXPLAIN ANALYZE
SELECT
  COUNT(*)
FROM
  incydenty i
WHERE
  i.imie_wroga IN (
    SELECT
      imie_wroga
    FROM
      wrogowie_ref
  )

In [ ]:
%%sql pgkotki
-- P.7: wariant b) zwykly JOIN:
EXPLAIN ANALYZE
SELECT
  COUNT(*)
FROM
  incydenty i
  JOIN wrogowie_ref w ON w.imie_wroga = i.imie_wroga

In [ ]:
%%sql pgkotki
-- P.7: wariant c) JOIN + COUNT(DISTINCT):
EXPLAIN ANALYZE
SELECT
  COUNT(DISTINCT i.id)
FROM
  incydenty i
  JOIN wrogowie_ref w ON w.imie_wroga = i.imie_wroga

Plany dla wariantów (1) oraz (2) są **identyczne** — optymalizator automatycznie przekształca predykat `IN` z podzapytaniem w półzłączenie (`Hash Semi Join`), uzyskując ten sam czas wykonania (~290 ms wobec ~295 ms, różnica w granicach szumu pomiarowego). Ręczne przepisywanie podzapytania `IN` na zwykły `JOIN` nie przynosi zatem żadnej korzyści wydajnościowej.

Szczególną uwagę należy zwrócić na wariant (3): użycie `COUNT(DISTINCT i.id)` wymusza dodatkowy etap deduplikacji nad wynikiem złączenia, co wprowadza do planu kosztowne sortowanie na dysku (`Sort` z `external merge Disk: 11768kB`, czas wykonania ~450 ms). Wariant „zoptymalizowany” zgodnie z poradą okazuje się najwolniejszy, ponieważ wykonuje zbędną pracę eliminacji duplikatów, których półzłączenie w wariancie (1) w ogóle nie generuje.

**Werdykt: MIT.** Ponadto bezrefleksyjne zastąpienie `IN` przez zwykły `JOIN` (bez `DISTINCT`) może **zniekształcić wynik zapytania**: jeśli w prawej tabeli występuje wiele dopasowań dla jednego klucza, złączenie wygeneruje duplikaty wierszy lewej tabeli (tzw. efekt *fan-out*), zawyżając wynik funkcji `COUNT(*)`. Do badania istnienia dopasowań służą operatory `IN` oraz `EXISTS`, natomiast `JOIN` służy do łączenia atrybutów z obu relacji.

### P.8 — „`UNION ALL` zamiast `UNION`, jeśli duplikaty nie grają roli"

**Rada (GfG + Medium):** `UNION` deduplikuje cały wynik — ten krok
kosztuje; jeśli zduplikowane wiersze ci nie przeszkadzają, pisz
`UNION ALL`.

**Zanim wykonasz:** obie gałęzie zwracają po pół miliona wierszy
(pseudonim + etykieta źródła). `UNION` ma je deduplikować — jak bardzo
to boli — zobaczysz w planie wyraźniej niż w pomiarach. Zapowiedz oba kształty.


In [ ]:
%%sql pgkotki
-- P.8: UNION vs UNION ALL -- cena deduplikacji milion wierszy
-- wariant a) UNION (usuwa duplikaty):
EXPLAIN
SELECT
  pseudo,
  'a' AS zrodlo
FROM
  incydenty
WHERE
  id <= 500000
UNION
SELECT
  pseudo,
  'b'
FROM
  incydenty
WHERE
  id > 500000

In [ ]:
%%sql pgkotki
-- P.8: wariant b) UNION ALL (dokleja bez deduplikacji):
EXPLAIN
SELECT
  pseudo,
  'a' AS zrodlo
FROM
  incydenty
WHERE
  id <= 500000
UNION ALL
SELECT
  pseudo,
  'b'
FROM
  incydenty
WHERE
  id > 500000

Plany wykonania jednoznacznie ilustrują różnicę między obydwoma operatorami. `UNION ALL` składa się wyłącznie z dwóch pełnych skanów (`Seq Scan`) połączonych węzłem `Append` — silnik wykonawczy bezpośrednio łączy strumienie danych bez dodatkowego przetwarzania. W przypadku `UNION` nad węzłem `Append` pojawia się operator `HashAggregate` (z 32 partycjami), ponieważ usunięcie duplikatów z miliona wierszy wymaga pełnego haszowania i buforowania zbioru wynikowego.

**Werdykt: PRAWDA.** Operator `UNION` wykonuje realną, kosztowną pracę deduplikacji całego wyniku (nawet wówczas, gdy gałęzie zapytania są z definicji rozłączne, jak w powyższym przykładzie ze stałymi etykietami `'a'` oraz `'b'`), podczas gdy `UNION ALL` pomija ten etap. Różnica ta jest bezpośrednio widoczna w strukturze i koszcie planu wykonania.

Należy jednak pamiętać o **różnicy semantycznej**: `UNION` realizuje sumę **zbiorów** (eliminując powtarzające się krotki), natomiast `UNION ALL` operuje na **multizbiorach** (ang. *multiset* / *bag*), zachowując wszystkie wiersze. Zamiana `UNION` na `UNION ALL` jest dopuszczalna wyłącznie wtedy, gdy duplikaty nie występują z natury danych lub ich obecność w wyniku jest akceptowalna biznesowo.

## Podsumowanie: tablica werdyktów

| # | Analizowana reguła | Werdykt | Weryfikacja w planie wykonania |
|---|---|---|---|
| P.1 | `GROUP BY` jest szybszy niż `DISTINCT` | **MIT** | identyczne plany 1:1 (ten sam operator `HashAggregate`) |
| P.2 | Unikaj podzapytań skorelowanych | **MIT** | `EXISTS` oraz `IN` optymalizator przekształca w półzłączenie (*Semi Join*) |
| P.3 | `NOT IN` działa tak samo jak `NOT EXISTS` | **MIT (niebezpieczny)** | `NOT IN` z wartością `NULL`: **0 wierszy**, `NOT EXISTS`: **902 020 wierszy** — różnica semantyczna (logika trójwartościowa) |
| P.4 | Unikaj `SELECT *` | **PRAWDA** | redukcja `width` z 38 do 17 B; zysk rośnie przy większych tablicach haszujących i sortowaniu |
| P.5 | `INNER JOIN` zamiast `LEFT JOIN`, gdy zbędny | **PRAWDA-ALE** | plany lustrzane (identyczny koszt `23 868`) — korzyść w postaci czytelności i jasnej intencji |
| P.6 | `LIKE 'x%'` szybkie, `LIKE '%x'` wolne | **PRAWDA-ALE** | bez indeksu: identyczny `Seq Scan` + `Filter` — reguła wymaga indeksu B-drzewa |
| P.7 | Przepisz `IN` na `JOIN` | **MIT** | `IN` i `JOIN` dają zbliżony czas (~290 ms); `JOIN + DISTINCT` jest **wolniejszy** przez dodatkową deduplikację |
| P.8 | `UNION ALL` zamiast `UNION` | **PRAWDA** | `UNION`: dodatkowy `HashAggregate` z 32 partycjami; `UNION ALL`: bezpośredni `Append` |

Wnioski ogólne wynikające z weryfikacji:

1. **Porady typu „przepisz składnię X na Y” są zazwyczaj mitami** (P.1, P.2, P.7) — współczesne optymalizatory przeprowadzają kanonizację i dekorelację zapytań automatycznie, natomiast ręczne przepisywanie bywa źródłem błędów semantycznych (P.3, P.7).
2. **Porady nakazujące redukcję zakresu przetwarzanych danych są prawdziwe** (P.4, P.8) — węższa projekcja oraz rezygnacja ze zbędnej deduplikacji bezpośrednio zmniejszają obciążenie pamięci i procesora.
3. **Reguły zależne od indeksów nie przynoszą efektu bez odpowiednich struktur fizycznych** (P.6).
4. **Poprawność semantyczna ma bezwzględny priorytet nad wydajnością** (P.3, P.8) — każdą optymalizację należy rozpoczynać od upewnienia się, że zmodyfikowane zapytanie zwraca identyczny wynik logiczny.

Warto dodać, że wprowadzenie **indeksów** zmienia przestrzeń dostępnych planów wykonania (np. umożliwia zastąpienie skanów sekwencyjnych operacjami `Index Scan` oraz zmianę algorytmów złączeń na `Nested Loops`). Dlatego przedstawione wnioski należy zawsze odnosić do konkretnej struktury fizycznej bazy danych oraz statystyk rozkładu wartości.

⏳ **Zadania praktyczne** — najpierw przeanalizuj problem teoretycznie i zapisz swoje przewidywania, a dopiero potem uruchom zapytanie w bazie danych.

---

### Z1 — Przewidywanie planu: dwa zapytania na bazie Chinook (PostgreSQL)

Dla każdego z poniższych zapytań, **przed jego uruchomieniem**:

1. naszkicuj przewidywany kształt drzewa operatorów (`Seq Scan`, `Hash Join`, `Sort`, itp.),
2. wskaż miejsca, w których optymalizator korzysta ze statystyk kolumnowych (`pg_stats`),
3. oceń, które zapytanie będzie bardziej kosztowne i uzasadnij dlaczego.

**Z1a** — 5 najczęściej kupowanych albumów:

```sql
EXPLAIN ANALYZE
SELECT a.title, COUNT(*) AS sprzedanych
FROM album a
JOIN track t          ON t.album_id = a.album_id
JOIN invoice_line il   ON il.track_id = t.track_id
GROUP BY a.title
ORDER BY sprzedanych DESC
LIMIT 5;
```

**Z1b** — klienci, którzy zakupili utwór z gatunku Blues, z wyłączeniem osób, które kupiły jakikolwiek utwór z gatunku Metal:

```sql
EXPLAIN ANALYZE
SELECT COUNT(DISTINCT i.customer_id) AS klientow
FROM invoice i
JOIN invoice_line il ON il.invoice_id = il.invoice_id
JOIN track t        ON t.track_id = il.track_id
JOIN genre g        ON g.genre_id = t.genre_id
WHERE g.name = 'Blues'
  AND i.customer_id NOT IN (
        SELECT i2.customer_id
        FROM invoice i2
        JOIN invoice_line il2 ON il2.invoice_id = i2.invoice_id
        JOIN track t2        ON t2.track_id = il2.track_id
        JOIN genre g2        ON g2.genre_id = t2.genre_id
        WHERE g2.name = 'Metal');
```

> **Pytanie kontrolne:** w zapytaniu Z1b zastosowano konstrukcję `NOT IN`. Odwołując się do wniosków z punktu P.3 — jakie zagrożenie niesie ten zapis w sytuacji, gdyby kolumna `customer_id` w podzapytaniu mogła przyjmować wartość `NULL`? Dlaczego `NOT EXISTS` stanowi bezpieczniejszą alternatywę i jak należałoby zmodyfikować zapytanie?

### Z2 — `EXISTS` kontra `JOIN + DISTINCT` pod filtrem

Porównamy dwa zapisy o identycznej semantyce logicznej.

**Zadanie:**

1. Przewidź, który wariant będzie miał niższy koszt szacunkowy oraz czy różnica w koszcie przełoży się na rzeczywisty czas wykonania.
2. Uruchom oba zapytania; porównaj `cost`, `Execution Time` oraz strukturę drzewa operatorów.
3. Uzasadnij, dlaczego jeden z tych zapisów jest preferowany w praktyce.

*(Jeśli dla małego słownika różnica czasów będzie niewielka, zastanów się, jak zachowają się oba plany przy większym zbiorze danych lub mniejszej selektywności filtru.)*

```sql
-- wariant A: EXISTS + selektywny filtr (typu z J.6)
EXPLAIN ANALYZE
SELECT COUNT(*) AS bez_filtru_a
FROM incydenty i
WHERE i.kraj = 'US'
  AND EXISTS (SELECT 1 FROM wrogowie_ref w
              WHERE w.imie_wroga = i.imie_wroga);

-- wariant B: JOIN + DISTINCT pod tym samym filtrem
EXPLAIN ANALYZE
SELECT COUNT(DISTINCT i.id) AS bez_filtru_b
FROM incydenty i
JOIN wrogowie_ref w ON w.imie_wroga = i.imie_wroga
WHERE i.kraj = 'US';
```

> **Pytanie kontrolne:** dlaczego w wariancie B użycie `COUNT(DISTINCT …)` jest konieczne dla zachowania poprawności wyniku, podczas gdy w wariancie A wystarcza zwykłe `COUNT(*)`? Co stałoby się z wynikiem wariantu B po zastąpieniu `COUNT(DISTINCT i.id)` przez `COUNT(*)`, gdyby po prawej stronie złączenia występowało wiele dopasowań dla jednego klucza?

### Z3 — Unikanie sortowania miliona wierszy (przepełnienie pamięci i redukcja strumienia)

Raport ma zwrócić **listę unikalnych pseudonimów kotów posortowaną alfabetycznie** (20 rekordów). Naiwna implementacja pobiera wszystkie wiersze z tabeli, pozostawiając deduplikację warstwie aplikacji. Prowadzi to do konieczności sortowania całego milionowego zbioru — operator `Sort` ponosi koszt przetwarzania każdego wiersza wejściowego.

**Zadanie:**

1. Wykonaj zapytanie Z3a i przeanalizuj węzeł `Sort`: ile wierszy podlega sortowaniu, jaką metodę zastosował silnik i czy nastąpiło przepełnienie na dysk (`Sort Method: external merge … Disk`)? Zanotuj czas wykonania.
2. Zapisz zapytanie Z3b tak, aby sortowanie odbywało się dopiero po **redukcji duplikatów** (na 20 wierszach), zachowując wymaganą semantykę raportu (unikalne pseudonimy uporządkowane alfabetycznie).
3. Wykonaj swoją wersję i porównaj plany: na którym etapie nastąpiła redukcja oraz jak zmieniły się `Sort Method` i czas wykonania?

```sql
-- Z3a: naiwna wersja — duplikaty odfiltrowuje aplikacja
EXPLAIN ANALYZE
SELECT pseudo FROM incydenty ORDER BY pseudo;

-- Z3b: twoja wersja — zredukuj strumień przed sortem
```

> **Pytanie kontrolne:** dlaczego eliminacja duplikatów w wariancie Z3b wykonywana jest **przed** sortowaniem, mimo że w składni SQL klauzula `DISTINCT` zapisana jest na początku zapytania, a `ORDER BY` na jego końcu? Który komponent silnika decyduje o kolejności operatorów i w jakich sytuacjach (np. przy `COUNT(DISTINCT …)` po złączeniu, jak w punkcie P.7) redukcja nie może zostać przesunięta przed złączenie?

# Załącznik A — Rozwiązania zadań Z1–Z3

Rozwiązania należy analizować **po** podjęciu samodzielnej próby — służą one jako wzorzec poprawnego rozumowania diagnostycznego.

## A.1 — Z1: przewidywane plany wykonania (Chinook, PostgreSQL)

**Z1a — 5 najczęściej kupowanych albumów.** Kluczowe elementy planu:

- Złączenie trzech tabel (`album × track × invoice_line`) o umiarkowanych kardynalnościach realizowane jest za pomocą operatorów `Hash Join` z budową tablic haszujących dla mniejszych relacji (`album` — 347 wierszy, `track` — 3 503 wiersze), po których następuje agregacja.
- Klauzula `LIMIT 5` aktywuje wyspecjalizowany algorytm `Sort Method: top-N heapsort` — zamiast pełnego sortowania wszystkich 304 pogrupowanych albumów, silnik utrzymuje w pamięci wyłącznie 5-elementowy kopiec najlepszych wyników.
- Agregacja realizowana jest przez `HashAggregate` z parametrem `Batches: 1` (wszystkie grupy mieszczą się bez problemu w pamięci operacyjnej).

**Z1b — Klienci kupujący Blues, lecz niekupujący Metalu.** Istotne cechy planu:

- Niewielkie rozmiary tabel w bazie Chinook sprawiają, że optymalizator chętnie wykorzystuje skany indeksowe (`Index Only Scan` / `Index Scan`) oraz złączenia `Nested Loop`.
- Podzapytanie w klauzuli `NOT IN` przekształcane jest w podplan typu `hashed SubPlan` (wynik podzapytania obliczany jest jednokrotnie i przechowywany w tablicy haszującej w pamięci). Zgodnie z wnioskami z punktu P.3 konstrukcja ta działa poprawnie wyłącznie dlatego, że kolumna `customer_id` nie zawiera wartości `NULL`; bezpieczniejszym i bardziej uniwersalnym zapisem jest skorelowane `NOT EXISTS`.

## A.2 — Z2: `EXISTS` kontra `JOIN + DISTINCT` pod filtrem

Oba zapytania zwracają identyczną liczbę unikalnych rekordów spełniających kryteria, jednak generują odmienne plany wykonania:

- **Wariant A (`EXISTS`, `COUNT(*)`):** wykorzystuje operator półzłączenia (`Hash Semi Join` lub `Nested Loop Semi Join`), który **nie powiela wierszy** lewej tabeli w przypadku wielu dopasowań po prawej stronie. Zwykłe `COUNT(*)` jest tu w pełni poprawne.
- **Wariant B (`JOIN + DISTINCT`, `COUNT(DISTINCT i.id)`):** wykonuje pełne złączenie wewnętrzne, a następnie wymusza dodatkowy etap deduplikacji (`Sort` / `Aggregate`) nad wynikiem złączenia.

**Zagrożenie:** zastąpienie `COUNT(DISTINCT i.id)` przez `COUNT(*)` w wariancie B doprowadziłoby do błędu merytorycznego w przypadku relacji jeden-do-wielu (efekt *fan-out* — pojedynczy wiersz z lewej tabeli zostałby policzony wielokrotnie). Półzłączenie w wariancie A eliminuje ryzyko duplikacji na poziomie samego operatora złączenia.

**Wniosek:** przy niewielkim słowniku po prawej stronie różnice czasowe mogą być nieznaczne, jednak przewaga `EXISTS` rośnie wyraźnie wraz ze wzrostem kardynalności i liczby dopasowań po stronie wewnętrznej.

## A.3 — Z3: sortowanie dwudziestu wierszy zamiast miliona

- **Z3a (`ORDER BY` bez deduplikacji):** operator `Sort` przetwarza cały milion wierszy, co skutkuje przepełnieniem pamięci (`Sort Method: external merge Disk: ~25 MB`) i czasem wykonania rzędu ~0,9 s.
- **Z3b (`SELECT DISTINCT pseudo FROM incydenty ORDER BY pseudo;` lub `GROUP BY pseudo`):** optymalizator umieszcza operator `HashAggregate` bezpośrednio nad skanem tabeli (`Seq Scan`), redukując strumień z miliona wierszy do 20 unikalnych grup **przed** operacją sortowania. W rezultacie węzeł `Sort` porządkuje zaledwie 20 rekordów w pamięci (`Sort Method: quicksort Memory: 25kB`), skracając czas wykonania kilkukrotnie i całkowicie eliminując operacje dyskowe.

**Wyjaśnienie kolejności operatorów:** kolejność zapisu klauzul w języku deklaratywnym SQL nie narzuca kolejności fizycznego wykonania. Optymalizator buduje drzewo algebraiczne, w którym eliminacja duplikatów (`δ` / `γ`) umieszczana jest poniżej sortowania (`τ`), co minimalizuje rozmiar sortowanego zbioru. Przesunięcie deduplikacji przed złączenie nie jest natomiast możliwe w przypadku `COUNT(DISTINCT …)` po złączeniu (jak w P.7), ponieważ zbiór kluczy do deduplikacji powstaje dopiero po połączeniu obu relacji.

# Załącznik B — Zestawienie weryfikowanych porad i wyników analizy

## B.1 — Źródła analizowanych praktyk

1. DataWithSantosh, *SQL Optimization Techniques — Enhancing Query Performance*, Medium:
   `https://medium.com/@DataWithSantosh/sql-optimization-techniques-enhancing-query-performance-f030f4c2c5f5`
2. GeeksforGeeks, *Best Practices for SQL Query Optimization*:
   `https://www.geeksforgeeks.org/sql/best-practices-for-sql-query-optimizations/`

Obie publikacje prezentują ogólne zalecenia bez odniesienia do statystyk danych, struktury indeksów oraz planów wykonania. W niniejszym tutorialu zastosowano podejście odwrotne: **weryfikację każdej reguły na podstawie planu wykonania**.

## B.2 — Tabela zbiorcza

| Reguła (skrót) | Źródło | Werdykt | Weryfikacja w planie wykonania |
|---|---|---|---|
| Indeksuj kolumny używane w `WHERE` i `JOIN` | obie | **PRAWDA** (temat osobnych zajęć) | `Seq Scan` vs `Index Scan` / `Index Seek` |
| „`GROUP BY` jest szybszy niż `DISTINCT`” | Medium | **MIT** | identyczne operatory `HashAggregate` (P.1) |
| „Unikaj podzapytań skorelowanych” | Medium | **MIT** (w PG 12+ i nowoczesnych SZBD) | brak `SubPlan` — dekorelacja do półzłączenia (P.2) |
| „`NOT IN` działa tak samo jak `NOT EXISTS`” | internet | **MIT (niebezpieczny)** | `SubPlan` vs `Hash Anti Join`; **0 vs 902 020 wierszy** przy obecności `NULL` (P.3) |
| „Unikaj `SELECT *`” | obie | **PRAWDA** | mniejsza wartość `width` skanu oraz niższe `Memory Usage` (P.4) |
| „`INNER JOIN` zamiast zbędnego `LEFT JOIN`” | GeeksforGeeks | **PRAWDA-ALE** (czytelność) | identyczne plany i koszt (P.5) |
| „`LIKE 'x%'` jest szybkie, `%x` wolne” | GeeksforGeeks | **PRAWDA-ALE** (wymaga indeksu) | bez indeksu: identyczny `Seq Scan` + `Filter` (P.6) |
| „`IN (SELECT …)` przepisz na `JOIN`” | GeeksforGeeks | **MIT** | `IN` = półzłączenie; `JOIN + DISTINCT` = zbędny koszt deduplikacji (P.7) |
| „`UNION ALL` zamiast `UNION`, gdy duplikaty nie grają roli” | obie | **PRAWDA** | `HashAggregate` nad `Append` vs czysty `Append` (P.8) |
| Ograniczaj liczbę zwracanych wierszy (`LIMIT` / `TOP`) | obie | **PRAWDA** | operator `Limit` aktywuje algorytm `top-N heapsort` (Z1a) |
| Partycjonowanie, buforowanie planów, procedury składowane | Medium | decyzje architektoniczne | poza zakresem pojedynczego zapytania SQL |
| Problem N+1 zapytań | GeeksforGeeks | problem warstwy aplikacji / ORM | wymaga agregacji lub złączenia po stronie SQL |
| „Analizuj plany wykonania zapytań” | obie | **PRAWDA (fundament diagnostyki)** | podstawa metodyki prezentowanej w całym tutorialu |

# Załącznik C — Metodologia pomiarów wydajności

## C.1 — Dlaczego `EXPLAIN ANALYZE`, a nie pomiar stoperem w aplikacji

Pomiar czasu wykonywany szacunkowo z poziomu klienta obejmuje wiele czynników zewnętrznych: opóźnienia sieciowe, narzut sterownika bazy danych, renderowanie wyników w interfejsie Jupyter oraz stan pamięci podręcznej. Polecenie `EXPLAIN ANALYZE` mierzy czas bezpośrednio wewnątrz silnika bazodanowego — każdy węzeł planu otrzymuje własny licznik `actual time`, a całkowity czas przetwarzania raportowany jest jako `Execution Time`. Ponadto plan dostarcza **metryk diagnostycznych niedostępnych z poziomu zegara**: porównania szacowanej i rzeczywistej liczby wierszy (`rows` vs `actual rows`), liczby iteracji (`loops`), zużycia pamięci (`Memory Usage`), liczby partii (`Batches`), operacji dyskowych (`Disk`) oraz wykorzystania buforów (`Buffers`).

## C.2 — Wiarygodność pojedynczego pomiaru

- **Pojedynczy pomiar bywa mylący:** różnice czasu wykonania poniżej ±30% w jednorazowym uruchomieniu wynikają najczęściej z szumu pomiarowego (pierwsze uruchomienie odczytuje strony z dysku do pamięci podręcznej *cache*, kolejne korzystają z danych w RAM; w tle działają procesy systemowe, takie jak *checkpoint* czy *autovacuum*).
- **Powtarzalność testów:** przy różnicach rzędu kilkunastu procent należy powtórzyć pomiar kilkukrotnie i zweryfikować, czy przewaga jednego wariantu utrzymuje się systematycznie.
- **Zmiany jakościowe:** w eksperymentach powodujących zmianę algorytmu lub przepełnienie pamięci na dysk (`work_mem`, *external merge*, partycjonowanie haszowe) różnice są skokowe (wielokrotne) i jednoznaczne diagnostycznie.
- **Determinizm modelu kosztowego:** wycena kosztu (`cost=`) jest powtarzalna przy niezmienionych statystykach — gdy czasy wykonania wykazują dużą wariancję, koszt wskazuje teoretyczną ocenę planera.

## C.3 — Czynniki wpływające na rozrzut pomiarów w środowisku laboratoryjnym

1. Tabela `incydenty` jest tabelą **tymczasową** (*temp schema*) — jej strony buforowane są w prywatnej pamięci podręcznej sesji (`local buffers`).
2. Pierwsze zapytania w bloku ładują strony tabeli do pamięci, przez co kolejne operacje na tym samym zbiorze mogą działać szybciej.
3. Opcja `TIMING ON` (domyślna w `EXPLAIN ANALYZE`) wprowadza niewielki narzut związany z odczytem zegara systemowego dla każdego wiersza w węzłach planu — przy bardzo krótkich czasach (rzędu pojedynczych milisekund) należy uwzględnić ten margines.
4. Środowisko laboratoryjne działa na współdzielonym serwerze bazodanowym — równoległa aktywność innych użytkowników może znacząco wpłynąć na chwilowe obciążenie procesora i podsystemu dyskowego.